In [1]:
# ============================================================
# S4 — ARCHITECTURE D + KNOWLEDGE DISTILLATION
# CELL 1: Environment setup, HF dataset preparation,
# data validation, and checkpoint discovery.
#
# Safe to run first in a fresh Kaggle notebook.
# No training; no Fold 3 gold labels are loaded or used.
# ============================================================

import os
import re
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import torch

print("=" * 78)
print("S4 — ARCHITECTURE D + KNOWLEDGE DISTILLATION")
print("CELL 1: SETUP AND PREFLIGHT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Workspace and hardware
# ------------------------------------------------------------
S4_WORK_ROOT = Path("/kaggle/working")
S4_INPUT_ROOT = Path("/kaggle/input")
S4_OUTPUT_DIR = S4_WORK_ROOT / "S4_Architecture_D_KD"
S4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S4_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\n[1] Environment")
print("PyTorch:", torch.__version__)
print("Device:", S4_DEVICE)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB",
    )

# ------------------------------------------------------------
# 2. Read Hugging Face token from Kaggle Secrets
# ------------------------------------------------------------
S4_HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get(
    "HUGGINGFACEHUB_API_TOKEN"
)

try:
    from kaggle_secrets import UserSecretsClient

    S4_SECRETS_CLIENT = UserSecretsClient()

    try:
        S4_SECRET_TOKEN = S4_SECRETS_CLIENT.get_secret("HF")
        if S4_SECRET_TOKEN and S4_SECRET_TOKEN.strip():
            S4_HF_TOKEN = S4_SECRET_TOKEN.strip()
    except Exception as exc:
        print("Kaggle Secret 'HF' could not be read:", str(exc)[:250])

except ImportError:
    print("Kaggle Secrets client unavailable; checking existing token variables.")

if S4_HF_TOKEN:
    print("Hugging Face token: available (value not displayed)")
else:
    print("Hugging Face token: not found")

# ------------------------------------------------------------
# 3. Locate or download the Phase 2 dataset
# ------------------------------------------------------------
S4_PHASE2_REPO_ID = "Exploration-Lab/CS-781-Capstone"
S4_DOWNLOAD_DIR = S4_WORK_ROOT / "phase2_lner"

S4_PHASE2_ROOT = None

# Prefer an already-mounted copy, including the previous S2 location.
S4_KNOWN_PHASE2_PATHS = [
    S4_INPUT_ROOT / "datasets" / "rohitbatra2004"
    / "cs781-capstone" / "Phase_2_LNER",

    S4_INPUT_ROOT / "datasets" / "rohitbatra2004"
    / "phase2-lner" / "Phase_2_LNER",

    S4_DOWNLOAD_DIR / "Phase_2_LNER",
    S4_WORK_ROOT / "phase2_lner" / "Phase_2_LNER",
]

for path in S4_KNOWN_PHASE2_PATHS:
    if (
        (path / "Train" / "fold1-text.json").is_file()
        and (path / "Test" / "fold3-text.json").is_file()
    ):
        S4_PHASE2_ROOT = path
        break

# Search mounted Kaggle inputs if the dataset is stored under another name.
if S4_PHASE2_ROOT is None and S4_INPUT_ROOT.exists():
    for marker in S4_INPUT_ROOT.rglob("fold1-text.json"):
        candidate = marker.parent.parent

        if (
            (candidate / "Train" / "fold2-text.json").is_file()
            and (candidate / "Test" / "fold3-text.json").is_file()
        ):
            S4_PHASE2_ROOT = candidate
            break

# Download only Phase_2_LNER files if no local copy exists.
if S4_PHASE2_ROOT is None:
    if not S4_HF_TOKEN:
        raise RuntimeError(
            "Phase 2 data was not found locally and Kaggle Secret 'HF' "
            "is unavailable. Add the Hugging Face token as a Kaggle Secret "
            "named HF, then rerun Cell 1."
        )

    print("\nDownloading Phase_2_LNER from Hugging Face...")

    from huggingface_hub import snapshot_download

    S4_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    snapshot_download(
        repo_id=S4_PHASE2_REPO_ID,
        repo_type="dataset",
        token=S4_HF_TOKEN,
        allow_patterns=["Phase_2_LNER/**"],
        local_dir=str(S4_DOWNLOAD_DIR),
    )

    S4_CANDIDATE = S4_DOWNLOAD_DIR / "Phase_2_LNER"

    if (
        (S4_CANDIDATE / "Train" / "fold1-text.json").is_file()
        and (S4_CANDIDATE / "Test" / "fold3-text.json").is_file()
    ):
        S4_PHASE2_ROOT = S4_CANDIDATE
    else:
        for marker in S4_DOWNLOAD_DIR.rglob("fold1-text.json"):
            candidate = marker.parent.parent

            if (
                (candidate / "Train" / "fold2-text.json").is_file()
                and (candidate / "Test" / "fold3-text.json").is_file()
            ):
                S4_PHASE2_ROOT = candidate
                break

if S4_PHASE2_ROOT is None:
    raise FileNotFoundError(
        "Phase 2 dataset was not found after discovery/download."
    )

S4_TRAIN_DIR = S4_PHASE2_ROOT / "Train"
S4_TEST_DIR = S4_PHASE2_ROOT / "Test"

S4_REQUIRED_DATA_FILES = [
    S4_TRAIN_DIR / "fold1-text.json",
    S4_TRAIN_DIR / "fold1-gold.json",
    S4_TRAIN_DIR / "fold2-text.json",
    S4_TRAIN_DIR / "fold2-gold.json",
    S4_TRAIN_DIR / "labels.txt",
    S4_TEST_DIR / "fold3-text.json",
]

for path in S4_REQUIRED_DATA_FILES:
    if not path.is_file():
        raise FileNotFoundError(f"Required Phase 2 file missing: {path}")

print("\n[2] Phase 2 dataset")
print("Dataset root:", S4_PHASE2_ROOT)
print("Source:", "mounted/local dataset" if S4_PHASE2_ROOT.is_relative_to(S4_INPUT_ROOT)
      else "working/download directory")

# ------------------------------------------------------------
# 4. Load and validate all permitted data
#    Do not open or search for a Fold 3 gold-label file.
# ------------------------------------------------------------
def s4_load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


S4_FOLD1_TEXT_RAW = s4_load_json(S4_TRAIN_DIR / "fold1-text.json")
S4_FOLD1_GOLD_RAW = s4_load_json(S4_TRAIN_DIR / "fold1-gold.json")
S4_FOLD2_TEXT_RAW = s4_load_json(S4_TRAIN_DIR / "fold2-text.json")
S4_FOLD2_GOLD_RAW = s4_load_json(S4_TRAIN_DIR / "fold2-gold.json")
S4_FOLD3_TEXT_RAW = s4_load_json(S4_TEST_DIR / "fold3-text.json")

S4_LABELS = [
    line.strip()
    for line in (S4_TRAIN_DIR / "labels.txt").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

# Normalize document IDs to strings.
S4_FOLD1_TEXT = {
    str(doc_id): text for doc_id, text in S4_FOLD1_TEXT_RAW.items()
}
S4_FOLD2_TEXT = {
    str(doc_id): text for doc_id, text in S4_FOLD2_TEXT_RAW.items()
}
S4_FOLD3_TEXT = {
    str(doc_id): text for doc_id, text in S4_FOLD3_TEXT_RAW.items()
}
S4_FOLD1_GOLD = {
    str(doc_id): annotations
    for doc_id, annotations in S4_FOLD1_GOLD_RAW.items()
}
S4_FOLD2_GOLD = {
    str(doc_id): annotations
    for doc_id, annotations in S4_FOLD2_GOLD_RAW.items()
}

for split_name, texts in [
    ("fold1", S4_FOLD1_TEXT),
    ("fold2", S4_FOLD2_TEXT),
    ("fold3", S4_FOLD3_TEXT),
]:
    invalid = [
        doc_id for doc_id, text in texts.items()
        if not isinstance(text, str)
    ]
    if invalid:
        raise TypeError(
            f"{split_name} contains non-string document text: "
            f"{invalid[:5]}"
        )

if set(S4_FOLD1_TEXT) != set(S4_FOLD1_GOLD):
    raise ValueError("Fold 1 text IDs and gold-annotation IDs differ.")

if set(S4_FOLD2_TEXT) != set(S4_FOLD2_GOLD):
    raise ValueError("Fold 2 text IDs and gold-annotation IDs differ.")

if set(S4_FOLD1_TEXT) & set(S4_FOLD2_TEXT):
    raise ValueError("Fold 1 and Fold 2 contain duplicate document IDs.")

S4_ALL_TRAIN_TEXTS = {**S4_FOLD1_TEXT, **S4_FOLD2_TEXT}
S4_ALL_TRAIN_GOLD = {**S4_FOLD1_GOLD, **S4_FOLD2_GOLD}

if set(S4_ALL_TRAIN_TEXTS) & set(S4_FOLD3_TEXT):
    raise ValueError("Training and Fold 3 document IDs overlap.")

if len(S4_LABELS) != 12 or len(set(S4_LABELS)) != 12:
    raise ValueError(
        f"Expected 12 unique Phase 2 labels, found {S4_LABELS}"
    )

print("\n[3] Dataset validation")
print("Fold 1 labeled documents:", len(S4_FOLD1_TEXT))
print("Fold 2 labeled documents:", len(S4_FOLD2_TEXT))
print("Total labeled documents:", len(S4_ALL_TRAIN_TEXTS))
print("Fold 3 test documents:", len(S4_FOLD3_TEXT))
print("Number of entity labels:", len(S4_LABELS))
print("Labels:", S4_LABELS)
print("Fold 3 gold labels loaded: NO")

# ------------------------------------------------------------
# 5. Discover Architecture D Phase 1 model files
# ------------------------------------------------------------
S4_DISCOVERY_ROOTS = [
    root for root in (S4_INPUT_ROOT, S4_WORK_ROOT)
    if root.exists()
]

S4_ARCH_D_CHECKPOINT_CANDIDATES = []
S4_ARCH_D_METADATA_CANDIDATES = []
S4_TOKENIZER_CANDIDATES = []

for root in S4_DISCOVERY_ROOTS:
    for path in root.rglob("*"):
        if not path.is_file():
            continue

        low_name = path.name.lower()
        low_path = str(path).lower()

        if path.suffix.lower() in {".pt", ".pth", ".safetensors"}:
            if (
                "model_d" in low_name
                or "architecture_d" in low_path
                or "architecture-d" in low_path
                or "arch_d" in low_path
            ):
                S4_ARCH_D_CHECKPOINT_CANDIDATES.append(path)

        if path.name.lower() in {
            "metadata.json",
            "model_config.json",
            "config.json",
        } and (
            "architecture_d" in low_path
            or "architecture-d" in low_path
            or "arch_d" in low_path
            or "model_d" in low_path
        ):
            S4_ARCH_D_METADATA_CANDIDATES.append(path)

        if low_name == "tokenizer.json":
            S4_TOKENIZER_CANDIDATES.append(path)

# Prioritize the established shared tokenizer location if present.
S4_TOKENIZER_CANDIDATES.sort(
    key=lambda path: (
        "cs781-phase1-tokenized-corpus" not in str(path).lower(),
        len(str(path)),
    )
)

# Deduplicate paths.
S4_ARCH_D_CHECKPOINT_CANDIDATES = list(
    dict.fromkeys(S4_ARCH_D_CHECKPOINT_CANDIDATES)
)
S4_ARCH_D_METADATA_CANDIDATES = list(
    dict.fromkeys(S4_ARCH_D_METADATA_CANDIDATES)
)
S4_TOKENIZER_CANDIDATES = list(dict.fromkeys(S4_TOKENIZER_CANDIDATES))

print("\n[4] Architecture D discovery")
print("Checkpoint candidates:", len(S4_ARCH_D_CHECKPOINT_CANDIDATES))
for path in S4_ARCH_D_CHECKPOINT_CANDIDATES[:20]:
    print(f"  {path} ({path.stat().st_size / 1024**2:.2f} MiB)")

print("Metadata candidates:", len(S4_ARCH_D_METADATA_CANDIDATES))
for path in S4_ARCH_D_METADATA_CANDIDATES[:20]:
    print(" ", path)

print("Tokenizer candidates:", len(S4_TOKENIZER_CANDIDATES))
for path in S4_TOKENIZER_CANDIDATES[:10]:
    print(" ", path)

# ------------------------------------------------------------
# 6. Discover a possible S3/Architecture C teacher checkpoint
# ------------------------------------------------------------
S4_TEACHER_CHECKPOINT_CANDIDATES = []
S4_TEACHER_ARCHIVE_CANDIDATES = []

for root in S4_DISCOVERY_ROOTS:
    for path in root.rglob("*"):
        if not path.is_file():
            continue

        low_name = path.name.lower()
        low_path = str(path).lower()

        is_s3_named = (
            "s3" in low_name
            or "architecture_c" in low_path
            or "architecture-c" in low_path
            or "arch_c" in low_path
            or "lorma" in low_name
        )

        if is_s3_named and path.suffix.lower() in {
            ".pt", ".pth", ".safetensors"
        }:
            S4_TEACHER_CHECKPOINT_CANDIDATES.append(path)

        if is_s3_named and path.suffix.lower() == ".zip":
            S4_TEACHER_ARCHIVE_CANDIDATES.append(path)

S4_TEACHER_CHECKPOINT_CANDIDATES = list(
    dict.fromkeys(S4_TEACHER_CHECKPOINT_CANDIDATES)
)
S4_TEACHER_ARCHIVE_CANDIDATES = list(
    dict.fromkeys(S4_TEACHER_ARCHIVE_CANDIDATES)
)

print("\n[5] Potential S3 teacher discovery")
print("Standalone checkpoint candidates:",
      len(S4_TEACHER_CHECKPOINT_CANDIDATES))
for path in S4_TEACHER_CHECKPOINT_CANDIDATES[:20]:
    print(f"  {path} ({path.stat().st_size / 1024**2:.2f} MiB)")

print("S3 archive candidates:", len(S4_TEACHER_ARCHIVE_CANDIDATES))
for path in S4_TEACHER_ARCHIVE_CANDIDATES[:20]:
    print(f"  {path} ({path.stat().st_size / 1024**2:.2f} MiB)")

# Inspect archive member names only; do not extract or load arbitrary files.
for archive_path in S4_TEACHER_ARCHIVE_CANDIDATES[:5]:
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            members = archive.namelist()
        print(f"\nArchive contents: {archive_path}")
        for member in members[:30]:
            print("  ", member)
        if len(members) > 30:
            print(f"  ... {len(members) - 30} additional entries")
    except (zipfile.BadZipFile, OSError) as exc:
        print("Could not inspect archive:", archive_path, str(exc)[:200])

# ------------------------------------------------------------
# 7. Save a preflight manifest (never save the HF token)
# ------------------------------------------------------------
S4_PREFLIGHT = {
    "experiment": "S4 Architecture D + Knowledge Distillation",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "phase2_root": str(S4_PHASE2_ROOT),
    "train_documents": len(S4_ALL_TRAIN_TEXTS),
    "fold3_documents": len(S4_FOLD3_TEXT),
    "labels": S4_LABELS,
    "fold3_gold_labels_loaded": False,
    "architecture_d_checkpoints": [
        str(path) for path in S4_ARCH_D_CHECKPOINT_CANDIDATES
    ],
    "architecture_d_metadata": [
        str(path) for path in S4_ARCH_D_METADATA_CANDIDATES
    ],
    "tokenizer_candidates": [
        str(path) for path in S4_TOKENIZER_CANDIDATES
    ],
    "teacher_checkpoints": [
        str(path) for path in S4_TEACHER_CHECKPOINT_CANDIDATES
    ],
    "teacher_archives": [
        str(path) for path in S4_TEACHER_ARCHIVE_CANDIDATES
    ],
    "hf_token_available": bool(S4_HF_TOKEN),
}

S4_PREFLIGHT_PATH = S4_OUTPUT_DIR / "S4_preflight_manifest.json"

with open(S4_PREFLIGHT_PATH, "w", encoding="utf-8") as file:
    json.dump(S4_PREFLIGHT, file, indent=2, ensure_ascii=False)

print("\n" + "=" * 78)
print("CELL 1 PREFLIGHT COMPLETE")
print("=" * 78)
print("Preflight manifest:", S4_PREFLIGHT_PATH)
print("Architecture D checkpoint found:",
      bool(S4_ARCH_D_CHECKPOINT_CANDIDATES))
print("S3 teacher checkpoint found:",
      bool(S4_TEACHER_CHECKPOINT_CANDIDATES))
print("Tokenizer found:", bool(S4_TOKENIZER_CANDIDATES))
print("HF token saved to disk: NO")
print("Fold 3 gold labels used: NO")

if not S4_ARCH_D_CHECKPOINT_CANDIDATES:
    print(
        "\nWARNING: No Architecture D checkpoint was discovered. "
        "Do not proceed to model construction until the correct "
        "Phase 1 Architecture D artifact is available."
    )

if not S4_TEACHER_CHECKPOINT_CANDIDATES and not S4_TEACHER_ARCHIVE_CANDIDATES:
    print(
        "\nNOTE: No S3 teacher checkpoint/archive was found by filename. "
        "We will determine whether it can be recovered from existing "
        "artifacts before selecting the distillation teacher."
    )

S4 — ARCHITECTURE D + KNOWLEDGE DISTILLATION
CELL 1: SETUP AND PREFLIGHT

[1] Environment
PyTorch: 2.11.0+cu128
Device: cuda
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB
Hugging Face token: available (value not displayed)



Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


[2] Phase 2 dataset
Dataset root: /kaggle/working/phase2_lner/Phase_2_LNER
Source: working/download directory

[3] Dataset validation
Fold 1 labeled documents: 35
Fold 2 labeled documents: 35
Total labeled documents: 70
Fold 3 test documents: 35
Number of entity labels: 12
Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Fold 3 gold labels loaded: NO

[4] Architecture D discovery
Checkpoint candidates: 3
  /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/checkpoints/latest_backup.pt (1306.50 MiB)
  /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/checkpoints/latest.pt (1306.50 MiB)
  /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/checkpoints/best.pt (1306.50 MiB)
Metadata candidates: 0
Tokenizer candidates: 1
  /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json

[5] Potential S3 teacher discovery
Standalone checkpoint candidates: 0
S3 arch

In [2]:
# ============================================================
# S4 — ARCHITECTURE D + KNOWLEDGE DISTILLATION
# CELL 2: Inspect Architecture D configuration and checkpoint.
#
# Uses only paths discovered in Cell 1.
# No training, no checkpoint modification, no Fold 3 gold labels.
# ============================================================

import json
import torch
from pathlib import Path

print("=" * 78)
print("S4 — ARCHITECTURE D CHECKPOINT INSPECTION")
print("=" * 78)

# ------------------------------------------------------------
# 1. Resolve Architecture D root from discovered checkpoint
# ------------------------------------------------------------
if not S4_ARCH_D_CHECKPOINT_CANDIDATES:
    raise RuntimeError(
        "No Architecture D checkpoint was discovered in Cell 1."
    )

S4_ARCH_D_BEST_PATH = next(
    (
        p for p in S4_ARCH_D_CHECKPOINT_CANDIDATES
        if p.name.lower() == "best.pt"
    ),
    None,
)

if S4_ARCH_D_BEST_PATH is None:
    S4_ARCH_D_BEST_PATH = S4_ARCH_D_CHECKPOINT_CANDIDATES[0]

# Expected structure:
# architecture_D/checkpoints/best.pt
S4_ARCH_D_ROOT = S4_ARCH_D_BEST_PATH.parent.parent.resolve()

if not (S4_ARCH_D_ROOT / "checkpoints").is_dir():
    raise RuntimeError(
        f"Unexpected Architecture D layout: {S4_ARCH_D_ROOT}"
    )

S4_ARCH_D_CONFIG_DIR = S4_ARCH_D_ROOT / "configs"

print("\n[1] Resolved paths")
print("Architecture D root:", S4_ARCH_D_ROOT)
print("Checkpoint selected:", S4_ARCH_D_BEST_PATH)
print(
    "Checkpoint size:",
    f"{S4_ARCH_D_BEST_PATH.stat().st_size / 1024**3:.2f} GiB",
)
print("Config directory exists:", S4_ARCH_D_CONFIG_DIR.is_dir())

# ------------------------------------------------------------
# 2. List files within the Architecture D artifact directory
# ------------------------------------------------------------
print("\n[2] Files under Architecture D")

S4_ARCH_D_FILES = sorted(
    p for p in S4_ARCH_D_ROOT.rglob("*")
    if p.is_file()
)

for path in S4_ARCH_D_FILES[:180]:
    try:
        relative = path.relative_to(S4_ARCH_D_ROOT)
    except ValueError:
        relative = path

    print(f"  {relative} [{path.stat().st_size:,} bytes]")

if len(S4_ARCH_D_FILES) > 180:
    print(f"  ... {len(S4_ARCH_D_FILES) - 180} additional files")

# ------------------------------------------------------------
# 3. Read small config/metadata files
# ------------------------------------------------------------
print("\n[3] Configuration files")

S4_ARCH_D_CONFIG_CONTENTS = {}
S4_ARCH_D_CONFIG_EXTENSIONS = {
    ".json", ".yaml", ".yml", ".toml", ".txt", ".md"
}

S4_ARCH_D_CONFIG_FILES = [
    p for p in S4_ARCH_D_FILES
    if (
        p.parent == S4_ARCH_D_CONFIG_DIR
        or any(
            term in p.name.lower()
            for term in ("config", "metadata", "architecture", "model_info")
        )
    )
    and p.suffix.lower() in S4_ARCH_D_CONFIG_EXTENSIONS
    and p.stat().st_size <= 1_000_000
]

for path in S4_ARCH_D_CONFIG_FILES[:30]:
    relative = str(path.relative_to(S4_ARCH_D_ROOT))

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    except Exception as exc:
        print(f"Could not read {relative}: {exc}")
        continue

    S4_ARCH_D_CONFIG_CONTENTS[relative] = content

    print("\n" + "-" * 72)
    print("FILE:", relative)
    print("-" * 72)
    print(content[:7000])

    if len(content) > 7000:
        print("... [display truncated]")

if not S4_ARCH_D_CONFIG_FILES:
    print("No readable config/metadata text files found.")

# ------------------------------------------------------------
# 4. Load checkpoint metadata on CPU using memory mapping
# ------------------------------------------------------------
# mmap=True avoids eagerly copying all tensor storage into RAM.
# This trusted Phase 1 artifact is inspected on CPU only.
print("\n[4] Inspecting best.pt")

try:
    S4_ARCH_D_CHECKPOINT_OBJECT = torch.load(
        S4_ARCH_D_BEST_PATH,
        map_location="cpu",
        mmap=True,
        weights_only=False,
    )
except Exception as exc:
    raise RuntimeError(
        "Unable to inspect the Architecture D checkpoint. "
        f"{type(exc).__name__}: {exc}"
    ) from exc


def s4_describe_checkpoint_object(obj, path="checkpoint", depth=0):
    """Describe nested checkpoint metadata without printing tensor values."""
    indent = "  " * depth

    if isinstance(obj, torch.Tensor):
        return {
            "type": "Tensor",
            "shape": list(obj.shape),
            "dtype": str(obj.dtype),
            "numel": int(obj.numel()),
        }

    if isinstance(obj, dict):
        print(f"{indent}{path}: dict ({len(obj)} keys)")
        summary = {}

        for key, value in obj.items():
            key_string = str(key)

            if isinstance(value, torch.Tensor):
                info = {
                    "type": "Tensor",
                    "shape": list(value.shape),
                    "dtype": str(value.dtype),
                    "numel": int(value.numel()),
                }
                summary[key_string] = info
                print(
                    f"{indent}  {key_string}: Tensor "
                    f"shape={tuple(value.shape)}, dtype={value.dtype}, "
                    f"numel={value.numel():,}"
                )

            elif isinstance(value, dict):
                summary[key_string] = s4_describe_checkpoint_object(
                    value,
                    f"{path}.{key_string}",
                    depth + 1,
                )

            elif isinstance(value, (str, int, float, bool, type(None))):
                summary[key_string] = {
                    "type": type(value).__name__,
                    "value": value if isinstance(
                        value, (str, int, float, bool, type(None))
                    ) else str(value),
                }
                print(
                    f"{indent}  {key_string}: "
                    f"{type(value).__name__} = {str(value)[:180]}"
                )

            elif isinstance(value, (list, tuple)):
                summary[key_string] = {
                    "type": type(value).__name__,
                    "length": len(value),
                }
                print(
                    f"{indent}  {key_string}: "
                    f"{type(value).__name__}, length={len(value)}"
                )

            else:
                summary[key_string] = {
                    "type": type(value).__name__,
                }
                print(
                    f"{indent}  {key_string}: "
                    f"{type(value).__name__}"
                )

        return summary

    return {
        "type": type(obj).__name__,
    }


S4_ARCH_D_CHECKPOINT_SUMMARY = (
    s4_describe_checkpoint_object(S4_ARCH_D_CHECKPOINT_OBJECT)
)

# ------------------------------------------------------------
# 5. Find nested model state dictionaries and count parameters
# ------------------------------------------------------------
def s4_find_tensor_dicts(obj, path="checkpoint"):
    """Locate nested dictionaries containing model-like tensor entries."""
    results = []

    if not isinstance(obj, dict):
        return results

    tensor_items = {
        str(key): value
        for key, value in obj.items()
        if isinstance(value, torch.Tensor)
    }

    if tensor_items:
        sample_keys = list(tensor_items.keys())[:10]
        model_like = any(
            any(token in key.lower() for token in (
                "embedding", "layers", "encoder", "decoder",
                "attention", "transformer", "lm_head",
            ))
            for key in tensor_items
        )

        results.append({
            "path": path,
            "tensor_count": len(tensor_items),
            "numel": int(sum(t.numel() for t in tensor_items.values())),
            "model_like_keys": model_like,
            "sample_keys": sample_keys,
            "tensor_dict": tensor_items,
        })

    for key, value in obj.items():
        if isinstance(value, dict):
            results.extend(
                s4_find_tensor_dicts(
                    value,
                    f"{path}.{key}",
                )
            )

    return results


S4_ARCH_D_TENSOR_DICTS = s4_find_tensor_dicts(
    S4_ARCH_D_CHECKPOINT_OBJECT
)

print("\n[5] Tensor dictionaries found:", len(S4_ARCH_D_TENSOR_DICTS))

for entry in sorted(
    S4_ARCH_D_TENSOR_DICTS,
    key=lambda item: item["numel"],
    reverse=True,
)[:12]:
    print(
        f"\n  Path: {entry['path']}\n"
        f"  Tensors: {entry['tensor_count']:,}\n"
        f"  Parameters/elements: {entry['numel']:,}\n"
        f"  Model-like keys: {entry['model_like_keys']}\n"
        f"  Sample keys: {entry['sample_keys']}"
    )

# ------------------------------------------------------------
# 6. Summarize Architecture D and save inspection report
# ------------------------------------------------------------
S4_ARCH_D_INSPECTION = {
    "architecture_root": str(S4_ARCH_D_ROOT),
    "selected_checkpoint": str(S4_ARCH_D_BEST_PATH),
    "checkpoint_size_bytes": S4_ARCH_D_BEST_PATH.stat().st_size,
    "device_used_for_inspection": "cpu",
    "memory_mapping_requested": True,
    "files": [
        {
            "relative_path": str(p.relative_to(S4_ARCH_D_ROOT)),
            "size_bytes": p.stat().st_size,
        }
        for p in S4_ARCH_D_FILES
    ],
    "config_files_read": list(S4_ARCH_D_CONFIG_CONTENTS.keys()),
    "checkpoint_summary": S4_ARCH_D_CHECKPOINT_SUMMARY,
    "tensor_dicts": [
        {
            key: value
            for key, value in entry.items()
            if key != "tensor_dict"
        }
        for entry in S4_ARCH_D_TENSOR_DICTS
    ],
    "fold3_gold_labels_used": False,
    "model_training_performed": False,
}

S4_ARCH_D_INSPECTION_PATH = (
    S4_OUTPUT_DIR / "S4_architecture_D_inspection.json"
)

with open(S4_ARCH_D_INSPECTION_PATH, "w", encoding="utf-8") as file:
    json.dump(
        S4_ARCH_D_INSPECTION,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("\n" + "=" * 78)
print("CELL 2 INSPECTION COMPLETE")
print("=" * 78)
print("Inspection report:", S4_ARCH_D_INSPECTION_PATH)
print("Checkpoint loaded on GPU: NO")
print("Training performed: NO")
print("Fold 3 gold labels used: NO")

S4 — ARCHITECTURE D CHECKPOINT INSPECTION

[1] Resolved paths
Architecture D root: /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D
Checkpoint selected: /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/checkpoints/best.pt
Checkpoint size: 1.28 GiB
Config directory exists: True

[2] Files under Architecture D
  .architecture_D_initialized [133 bytes]
  checkpoints/best.pt [1,369,959,633 bytes]
  checkpoints/latest.pt [1,369,960,715 bytes]
  checkpoints/latest_backup.pt [1,369,960,715 bytes]
  configs/data_manifest_D.json [3,012 bytes]
  configs/environment_manifest_D.json [699 bytes]
  configs/generation_engine_D.json [561 bytes]
  configs/generation_summary_D.json [913 bytes]
  configs/initial_sampler_state_D.json [1,190 bytes]
  configs/official_test_manifest_D.json [11,768 bytes]
  configs/runtime_configuration_D.json [20,023 bytes]
  configs/sampler_config_D.json [521 bytes]
  configs/selected_checkpoint_D.json [863 bytes]
  configs/submission_manifest_D.json

In [3]:
# ============================================================
# S4 — CELL 3: Find existing Phase 2 teacher checkpoints.
# Read-only file discovery; no model loading or training.
# ============================================================

from pathlib import Path

S4_TEACHER_SCAN_ROOTS = [
    Path("/kaggle/input/datasets"),
    Path("/kaggle/working"),
]

S4_TEACHER_KEYWORDS = (
    "archc", "architecture_c", "architecture-c",
    "lorma", "s3", "s2", "phase2", "phase_2",
    "ner", "trainable", "final_selected",
)

S4_TEACHER_FILE_CANDIDATES = []

for root in S4_TEACHER_SCAN_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()
        full_path = str(path).lower()

        if suffix not in {".pt", ".pth", ".safetensors", ".zip"}:
            continue

        relevant = any(word in full_path for word in S4_TEACHER_KEYWORDS)

        # Include generic best checkpoints inside Architecture C folders.
        if path.name.lower() in {"best.pt", "latest.pt"} and "archc" in full_path:
            relevant = True

        if relevant:
            S4_TEACHER_FILE_CANDIDATES.append(path)

S4_TEACHER_FILE_CANDIDATES = sorted(
    set(S4_TEACHER_FILE_CANDIDATES),
    key=lambda p: (p.suffix.lower() == ".zip", str(p).lower()),
)

print("=" * 70)
print("S4 — TEACHER ARTIFACT DISCOVERY")
print("=" * 70)

if not S4_TEACHER_FILE_CANDIDATES:
    print("No matching S2/S3/Architecture C artifacts found.")
else:
    for path in S4_TEACHER_FILE_CANDIDATES:
        print(
            f"{path} "
            f"({path.stat().st_size / 1024**2:.1f} MiB)"
        )

print(f"\nTotal candidates: {len(S4_TEACHER_FILE_CANDIDATES)}")
print("No files modified; no training performed.")

S4 — TEACHER ARTIFACT DISCOVERY
No matching S2/S3/Architecture C artifacts found.

Total candidates: 0
No files modified; no training performed.


In [4]:
# ============================================================
# S4 CELL 3 — Load Architecture D Phase 1 weights and verify
# strict checkpoint compatibility with a smoke test.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from tokenizers import Tokenizer

# Correct: checkpoint fields are at the top level.
S4_D_CKPT = S4_ARCH_D_CHECKPOINT_OBJECT
S4_D_CONFIG = S4_D_CKPT["model_config"]
S4_D_STATE = S4_D_CKPT["model_state_dict"]

S4_D_VOCAB = int(S4_D_CONFIG["vocab_size"])
S4_D_CHANNELS = int(S4_D_CONFIG["channels"])
S4_D_DILATIONS = list(S4_D_CONFIG["dilations"])
S4_D_KERNEL = int(S4_D_CONFIG["kernel_size"])
S4_D_DROPOUT = float(S4_D_CONFIG["dropout"])


class S4CausalConv(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.left_trim = (kernel_size - 1) * dilation
        self.conv = weight_norm(nn.Conv1d(
            channels, channels, kernel_size,
            padding=self.left_trim, dilation=dilation
        ))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        y = self.conv(x)
        if self.left_trim:
            y = y[..., :-self.left_trim]
        return self.dropout(F.relu(y))


class S4TCNBlock(nn.Module):
    def __init__(self, channels, kernel_size, dilation, dropout):
        super().__init__()
        self.conv1 = S4CausalConv(
            channels, kernel_size, dilation, dropout
        )
        self.conv2 = S4CausalConv(
            channels, kernel_size, dilation, dropout
        )

    def forward(self, x):
        return F.relu(x + self.conv2(self.conv1(x)))


class S4TCNLanguageModel(nn.Module):
    def __init__(
        self, vocab_size, channels, dilations, kernel_size, dropout
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, channels)
        self.tcn_blocks = nn.ModuleList([
            S4TCNBlock(channels, kernel_size, d, dropout)
            for d in dilations
        ])

    def encode(self, input_ids):
        x = self.token_embedding(input_ids).transpose(1, 2)
        for block in self.tcn_blocks:
            x = block(x)
        return x.transpose(1, 2)

    def forward(self, input_ids):
        hidden = self.encode(input_ids)
        return F.linear(hidden, self.token_embedding.weight)


S4_D_MODEL = S4TCNLanguageModel(
    vocab_size=S4_D_VOCAB,
    channels=S4_D_CHANNELS,
    dilations=S4_D_DILATIONS,
    kernel_size=S4_D_KERNEL,
    dropout=S4_D_DROPOUT,
)

# Strict loading: fail on any missing, unexpected, or incompatible keys.
S4_D_MODEL.load_state_dict(S4_D_STATE, strict=True)

S4_D_PARAM_COUNT = sum(p.numel() for p in S4_D_MODEL.parameters())
S4_D_EXPECTED_PARAMS = int(
    S4_D_CONFIG["estimated_parameter_count"]
)

if S4_D_PARAM_COUNT != S4_D_EXPECTED_PARAMS:
    raise RuntimeError(
        f"Parameter mismatch: {S4_D_PARAM_COUNT:,} vs "
        f"{S4_D_EXPECTED_PARAMS:,}"
    )

S4_TOKENIZER = Tokenizer.from_file(
    str(S4_TOKENIZER_CANDIDATES[0])
)

if S4_TOKENIZER.get_vocab_size() != S4_D_VOCAB:
    raise ValueError(
        f"Tokenizer vocab={S4_TOKENIZER.get_vocab_size()}, "
        f"model vocab={S4_D_VOCAB}"
    )

# Move model to the selected device and test a short training-text sample.
S4_D_MODEL = S4_D_MODEL.to(S4_DEVICE).eval()

S4_SMOKE_DOC_ID = next(iter(S4_ALL_TRAIN_TEXTS))
S4_SMOKE_IDS = S4_TOKENIZER.encode(
    S4_ALL_TRAIN_TEXTS[S4_SMOKE_DOC_ID]
).ids[:64]

if len(S4_SMOKE_IDS) < 2:
    raise RuntimeError("Smoke-test document produced too few tokens.")

S4_SMOKE_INPUT = torch.tensor(
    [S4_SMOKE_IDS],
    dtype=torch.long,
    device=S4_DEVICE,
)

with torch.inference_mode():
    S4_SMOKE_HIDDEN = S4_D_MODEL.encode(S4_SMOKE_INPUT)
    S4_SMOKE_LOGITS = S4_D_MODEL(S4_SMOKE_INPUT)

assert S4_SMOKE_HIDDEN.shape == (
    1, len(S4_SMOKE_IDS), S4_D_CHANNELS
)
assert S4_SMOKE_LOGITS.shape == (
    1, len(S4_SMOKE_IDS), S4_D_VOCAB
)
assert torch.isfinite(S4_SMOKE_HIDDEN).all()
assert torch.isfinite(S4_SMOKE_LOGITS).all()

print("Architecture:", S4_D_CONFIG["architecture_version"])
print(f"Parameters: {S4_D_PARAM_COUNT:,}")
print("Strict checkpoint loading: PASS")
print("Tokenizer vocabulary:", S4_TOKENIZER.get_vocab_size())
print("Hidden shape:", tuple(S4_SMOKE_HIDDEN.shape))
print("Logits shape:", tuple(S4_SMOKE_LOGITS.shape))
print("Finite outputs: PASS")
print("Training performed: NO")

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Architecture: D_v2
Parameters: 114,155,520
Strict checkpoint loading: PASS
Tokenizer vocabulary: 50000
Hidden shape: (1, 64, 1024)
Logits shape: (1, 64, 50000)
Finite outputs: PASS
Training performed: NO


In [5]:
# ============================================================
# S4 CELL 4 — Build character-aligned data windows, loaders,
# and the Architecture D character-span NER model.
# ============================================================

import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

S4_SEED = 42
random.seed(S4_SEED)
np.random.seed(S4_SEED)
torch.manual_seed(S4_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(S4_SEED)

S4_LABEL_TO_ID = {label: i for i, label in enumerate(S4_LABELS)}
S4_NUM_LABELS = len(S4_LABELS)

# ---------- Parse gold annotations robustly ----------
def s4_parse_annotations(raw_annotations, text, doc_id):
    if isinstance(raw_annotations, dict):
        for key in ("annotations", "entities", "spans"):
            if key in raw_annotations:
                raw_annotations = raw_annotations[key]
                break

    if not isinstance(raw_annotations, (list, tuple)):
        raise TypeError(
            f"Unsupported annotation container in document {doc_id}: "
            f"{type(raw_annotations).__name__}"
        )

    parsed = []
    for ann in raw_annotations:
        if isinstance(ann, dict):
            start = ann.get("start", ann.get("start_char", ann.get("start_character")))
            end = ann.get("end", ann.get("end_char", ann.get("end_character")))
            label = ann.get("label", ann.get("entity_label", ann.get("type")))
        elif isinstance(ann, (list, tuple)) and len(ann) >= 3:
            start, end, label = ann[0], ann[1], ann[2]
        else:
            raise ValueError(
                f"Unsupported annotation in {doc_id}: {str(ann)[:160]}"
            )

        if isinstance(start, bool) or isinstance(end, bool):
            raise TypeError(f"Boolean offsets in document {doc_id}.")
        start, end = int(start), int(end)

        if label not in S4_LABEL_TO_ID:
            raise ValueError(f"Unknown label {label!r} in {doc_id}.")
        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"Invalid annotation offsets in {doc_id}: {(start, end, label)}"
            )

        parsed.append((start, end, label))

    return sorted(parsed, key=lambda x: (x[0], x[1], S4_LABEL_TO_ID[x[2]]))


S4_PARSED_GOLD = {
    doc_id: s4_parse_annotations(
        S4_ALL_TRAIN_GOLD[doc_id],
        S4_ALL_TRAIN_TEXTS[doc_id],
        doc_id,
    )
    for doc_id in S4_ALL_TRAIN_TEXTS
}

S4_TRAIN_DOC_IDS, S4_VAL_DOC_IDS = train_test_split(
    sorted(S4_ALL_TRAIN_TEXTS),
    test_size=0.20,
    random_state=S4_SEED,
)

# Character vocabulary uses training documents only.
S4_CHAR_COUNTS = Counter(
    char
    for doc_id in S4_TRAIN_DOC_IDS
    for char in S4_ALL_TRAIN_TEXTS[doc_id]
)
S4_CHAR_TO_ID = {"<pad>": 0, "<unk>": 1}
for char in sorted(S4_CHAR_COUNTS):
    if char not in S4_CHAR_TO_ID:
        S4_CHAR_TO_ID[char] = len(S4_CHAR_TO_ID)
S4_UNK_CHAR_ID = S4_CHAR_TO_ID["<unk>"]

S4_WINDOW_TOKENS = 512
S4_WINDOW_STRIDE = 384


def s4_build_windows(doc_id, text, spans):
    """Tokenize a document and construct aligned character-supervision windows."""
    encoding = S4_TOKENIZER.encode(text, add_special_tokens=False)
    token_ids = encoding.ids
    offsets = encoding.offsets

    if not token_ids:
        if spans:
            raise ValueError(f"Document {doc_id} has spans but no tokens.")
        return [], set(), 0

    # Keep annotation IDs so we can confirm every gold span is covered.
    encoded_spans = [
        (start, end, S4_LABEL_TO_ID[label], idx)
        for idx, (start, end, label) in enumerate(spans)
    ]

    windows, covered, fallback_count = [], set(), 0
    token_start = 0

    while token_start < len(token_ids):
        token_end = min(token_start + S4_WINDOW_TOKENS, len(token_ids))
        local_offsets = offsets[token_start:token_end]

        valid = [
            (int(a), int(b), i)
            for i, (a, b) in enumerate(local_offsets)
            if int(b) > int(a)
        ]
        if not valid:
            raise RuntimeError(f"No valid token offsets in {doc_id}.")

        char_start = 0 if token_start == 0 else min(a for a, _, _ in valid)
        char_end = (
            len(text) if token_end == len(token_ids)
            else max(b for _, b, _ in valid)
        )
        if not (0 <= char_start < char_end <= len(text)):
            raise ValueError(f"Invalid character window in {doc_id}.")

        char_len = char_end - char_start
        char_to_token = np.full(char_len, -1, dtype=np.int16)

        for a, b, local_idx in valid:
            left = max(a, char_start) - char_start
            right = min(b, char_end) - char_start
            if right > left:
                char_to_token[left:right] = local_idx

        missing_positions = np.flatnonzero(char_to_token < 0)
        if len(missing_positions):
            starts = np.asarray([a for a, _, _ in valid])
            ends = np.asarray([b for _, b, _ in valid])
            for pos in missing_positions:
                absolute_pos = char_start + int(pos)
                containing = np.flatnonzero(
                    (starts <= absolute_pos) & (absolute_pos < ends)
                )
                if len(containing):
                    nearest = int(containing[0])
                else:
                    distance = np.where(
                        absolute_pos < starts,
                        starts - absolute_pos,
                        np.where(
                            absolute_pos >= ends,
                            absolute_pos - ends + 1,
                            0,
                        ),
                    )
                    nearest = int(np.argmin(distance))
                char_to_token[pos] = valid[nearest][2]
                fallback_count += 1

        char_ids = np.fromiter(
            (
                S4_CHAR_TO_ID.get(ch, S4_UNK_CHAR_ID)
                for ch in text[char_start:char_end]
            ),
            dtype=np.int32,
            count=char_len,
        )

        start_targets = np.zeros((char_len, S4_NUM_LABELS), dtype=np.float32)
        end_targets = np.zeros_like(start_targets)
        loss_mask = np.ones((char_len, S4_NUM_LABELS), dtype=np.bool_)

        for start, end, label_id, ann_idx in encoded_spans:
            if char_start <= start and end <= char_end:
                start_targets[start - char_start, label_id] = 1.0
                end_targets[end - 1 - char_start, label_id] = 1.0
                covered.add(ann_idx)
            else:
                if char_start <= start < char_end:
                    loss_mask[start - char_start, label_id] = False
                if char_start < end <= char_end:
                    loss_mask[end - 1 - char_start, label_id] = False

        windows.append({
            "doc_id": str(doc_id),
            "token_start": token_start,
            "char_start": char_start,
            "char_end": char_end,
            "input_ids": np.asarray(token_ids[token_start:token_end], dtype=np.int64),
            "char_ids": char_ids.astype(np.int64),
            "char_to_token": char_to_token.astype(np.int64),
            "start_targets": start_targets,
            "end_targets": end_targets,
            "loss_mask": loss_mask,
        })

        if token_end == len(token_ids):
            break
        token_start += S4_WINDOW_STRIDE

    return windows, covered, fallback_count


# Build all windows and verify all labeled entities are covered.
S4_WINDOWS_BY_DOC = {}
S4_TRAIN_WINDOWS, S4_VAL_WINDOWS = [], []
S4_ALIGNMENT_FALLBACKS = 0
S4_UNCOVERED_GOLD = []

for doc_id, text in S4_ALL_TRAIN_TEXTS.items():
    doc_windows, covered, fallbacks = s4_build_windows(
        doc_id, text, S4_PARSED_GOLD[doc_id]
    )
    S4_WINDOWS_BY_DOC[doc_id] = doc_windows
    S4_ALIGNMENT_FALLBACKS += fallbacks

    if len(covered) != len(S4_PARSED_GOLD[doc_id]):
        S4_UNCOVERED_GOLD.append({
            "doc_id": doc_id,
            "uncovered": len(S4_PARSED_GOLD[doc_id]) - len(covered),
        })

    if doc_id in S4_TRAIN_DOC_IDS:
        S4_TRAIN_WINDOWS.extend(doc_windows)
    else:
        S4_VAL_WINDOWS.extend(doc_windows)

if S4_UNCOVERED_GOLD:
    raise RuntimeError(
        "Some gold spans were not fully contained in any window: "
        f"{S4_UNCOVERED_GOLD[:10]}"
    )


# ---------- Collation ----------
def s4_collate_windows(items):
    batch_size = len(items)
    max_tokens = max(len(x["input_ids"]) for x in items)
    max_chars = max(len(x["char_ids"]) for x in items)

    input_ids = torch.zeros((batch_size, max_tokens), dtype=torch.long)
    char_ids = torch.zeros((batch_size, max_chars), dtype=torch.long)
    char_to_token = torch.zeros((batch_size, max_chars), dtype=torch.long)

    start_targets = torch.zeros(
        (batch_size, max_chars, S4_NUM_LABELS), dtype=torch.float32
    )
    end_targets = torch.zeros_like(start_targets)
    loss_mask = torch.zeros(
        (batch_size, max_chars, S4_NUM_LABELS), dtype=torch.bool
    )

    for i, item in enumerate(items):
        nt, nc = len(item["input_ids"]), len(item["char_ids"])
        input_ids[i, :nt] = torch.as_tensor(item["input_ids"])
        char_ids[i, :nc] = torch.as_tensor(item["char_ids"])
        char_to_token[i, :nc] = torch.as_tensor(
            item["char_to_token"].clip(0, max_tokens - 1)
        )
        start_targets[i, :nc] = torch.as_tensor(item["start_targets"])
        end_targets[i, :nc] = torch.as_tensor(item["end_targets"])
        loss_mask[i, :nc] = torch.as_tensor(item["loss_mask"])

    return {
        "input_ids": input_ids,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "start_targets": start_targets,
        "end_targets": end_targets,
        "loss_mask": loss_mask,
        "doc_ids": [x["doc_id"] for x in items],
        "char_starts": [x["char_start"] for x in items],
        "char_ends": [x["char_end"] for x in items],
    }


S4_BATCH_SIZE = 1
S4_TRAIN_LOADER = DataLoader(
    S4_TRAIN_WINDOWS,
    batch_size=S4_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=s4_collate_windows,
)
S4_VAL_LOADER = DataLoader(
    S4_VAL_WINDOWS,
    batch_size=S4_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=s4_collate_windows,
)


# ---------- Character-level start/end prediction head ----------
class S4TCNSpanNER(nn.Module):
    def __init__(self, backbone, char_vocab_size, num_labels, char_dim=32, hidden_dim=256):
        super().__init__()
        self.backbone = backbone
        self.char_embedding = nn.Embedding(
            char_vocab_size, char_dim, padding_idx=0
        )
        self.fuse = nn.Linear(
            S4_D_CHANNELS + char_dim, hidden_dim
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.conv1 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.15)
        self.start_head = nn.Linear(hidden_dim, num_labels)
        self.end_head = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids, char_ids, char_to_token):
        token_hidden = self.backbone.encode(input_ids)
        gather_idx = char_to_token.clamp(
            min=0, max=token_hidden.shape[1] - 1
        )
        char_hidden = torch.gather(
            token_hidden,
            1,
            gather_idx.unsqueeze(-1).expand(
                -1, -1, token_hidden.shape[-1]
            ),
        )
        char_features = self.char_embedding(char_ids)
        x = torch.cat([char_hidden, char_features], dim=-1)
        x = self.dropout(F.gelu(self.norm(self.fuse(x))))

        y = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x + self.dropout(y))
        y = self.conv2(x.transpose(1, 2)).transpose(1, 2)
        x = F.gelu(x + self.dropout(y))

        return self.start_head(x), self.end_head(x)


S4_TEACHER_MODEL = S4TCNSpanNER(
    backbone=S4_D_MODEL,
    char_vocab_size=len(S4_CHAR_TO_ID),
    num_labels=S4_NUM_LABELS,
).to(S4_DEVICE)

# Fourth-root balancing weights from training windows only.
S4_START_POS = np.zeros(S4_NUM_LABELS, dtype=np.float64)
S4_END_POS = np.zeros(S4_NUM_LABELS, dtype=np.float64)
S4_TOTAL_CHAR_POSITIONS = 0

for window in S4_TRAIN_WINDOWS:
    S4_START_POS += window["start_targets"].sum(axis=0)
    S4_END_POS += window["end_targets"].sum(axis=0)
    S4_TOTAL_CHAR_POSITIONS += len(window["char_ids"])

S4_POS_WEIGHTS = np.clip(
    (
        np.maximum(S4_TOTAL_CHAR_POSITIONS - S4_START_POS, 1)
        / np.maximum(S4_START_POS, 1)
    ) ** 0.25,
    2.0,
    12.0,
).astype(np.float32)

S4_POS_WEIGHTS_TENSOR = torch.tensor(
    S4_POS_WEIGHTS, device=S4_DEVICE, dtype=torch.float32
)

S4_CELL4_REPORT = {
    "train_documents": len(S4_TRAIN_DOC_IDS),
    "validation_documents": len(S4_VAL_DOC_IDS),
    "train_windows": len(S4_TRAIN_WINDOWS),
    "validation_windows": len(S4_VAL_WINDOWS),
    "train_gold_spans": sum(len(S4_PARSED_GOLD[d]) for d in S4_TRAIN_DOC_IDS),
    "validation_gold_spans": sum(len(S4_PARSED_GOLD[d]) for d in S4_VAL_DOC_IDS),
    "labels": S4_LABELS,
    "char_vocab_size": len(S4_CHAR_TO_ID),
    "alignment_fallbacks": S4_ALIGNMENT_FALLBACKS,
    "uncovered_gold_spans": 0,
}

with open(S4_OUTPUT_DIR / "S4_data_preprocessing_report.json", "w", encoding="utf-8") as f:
    json.dump(S4_CELL4_REPORT, f, indent=2, ensure_ascii=False)

S4_SMOKE_BATCH = next(iter(S4_TRAIN_LOADER))
with torch.no_grad():
    S4_SMOKE_START, S4_SMOKE_END = S4_TEACHER_MODEL(
        S4_SMOKE_BATCH["input_ids"].to(S4_DEVICE),
        S4_SMOKE_BATCH["char_ids"].to(S4_DEVICE),
        S4_SMOKE_BATCH["char_to_token"].to(S4_DEVICE),
    )

assert S4_SMOKE_START.shape == S4_SMOKE_BATCH["start_targets"].shape
assert S4_SMOKE_END.shape == S4_SMOKE_BATCH["end_targets"].shape
assert torch.isfinite(S4_SMOKE_START).all()
assert torch.isfinite(S4_SMOKE_END).all()

print("Train/validation docs:", len(S4_TRAIN_DOC_IDS), "/", len(S4_VAL_DOC_IDS))
print("Windows:", len(S4_TRAIN_WINDOWS), "train;", len(S4_VAL_WINDOWS), "validation")
print("Gold spans:", S4_CELL4_REPORT["train_gold_spans"], "train;",
      S4_CELL4_REPORT["validation_gold_spans"], "validation")
print("Character vocabulary:", len(S4_CHAR_TO_ID))
print("Alignment fallback count:", S4_ALIGNMENT_FALLBACKS)
print("Uncovered gold spans: 0")
print("Start/end logits shape:", tuple(S4_SMOKE_START.shape))
print("Cell 4 completed.")

Train/validation docs: 56 / 14
Windows: 903 train; 293 validation
Gold spans: 6532 train; 1861 validation
Character vocabulary: 159
Alignment fallback count: 0
Uncovered gold spans: 0
Start/end logits shape: (1, 2604, 12)
Cell 4 completed.


In [6]:
# ============================================================
# S4 CELL 5 — Supervised teacher training.
# Architecture D Phase 1 initialization; partial fine-tuning.
# Saves the best validation-loss checkpoint for later distillation.
# ============================================================

import json
import math
import time
import torch
import torch.nn.functional as F

S4_TEACHER_EPOCHS = 3
S4_GRAD_ACCUM = 3
S4_BACKBONE_LR = 2e-5
S4_HEAD_LR = 4e-4
S4_WEIGHT_DECAY = 0.01
S4_MAX_GRAD_NORM = 1.0

S4_TEACHER_CKPT_PATH = S4_OUTPUT_DIR / "S4_teacher_best.pt"
S4_TEACHER_HISTORY_PATH = S4_OUTPUT_DIR / "S4_teacher_history.json"
S4_TEACHER_CONFIG_PATH = S4_OUTPUT_DIR / "S4_teacher_config.json"

# Freeze the backbone, then unfreeze its last four residual TCN blocks.
for param in S4_TEACHER_MODEL.backbone.parameters():
    param.requires_grad = False

for block in S4_TEACHER_MODEL.backbone.tcn_blocks[6:]:
    for param in block.parameters():
        param.requires_grad = True

# Train the task-specific span head.
for module in (
    S4_TEACHER_MODEL.char_embedding,
    S4_TEACHER_MODEL.fuse,
    S4_TEACHER_MODEL.norm,
    S4_TEACHER_MODEL.conv1,
    S4_TEACHER_MODEL.conv2,
    S4_TEACHER_MODEL.start_head,
    S4_TEACHER_MODEL.end_head,
):
    for param in module.parameters():
        param.requires_grad = True

S4_BACKBONE_PARAMS = [
    p for p in S4_TEACHER_MODEL.backbone.parameters()
    if p.requires_grad
]
S4_HEAD_PARAMS = [
    p for name, p in S4_TEACHER_MODEL.named_parameters()
    if not name.startswith("backbone.") and p.requires_grad
]

S4_TRAINABLE_COUNT = sum(
    p.numel() for p in S4_TEACHER_MODEL.parameters()
    if p.requires_grad
)
S4_TOTAL_COUNT = sum(p.numel() for p in S4_TEACHER_MODEL.parameters())

S4_OPTIMIZER = torch.optim.AdamW(
    [
        {"params": S4_BACKBONE_PARAMS, "lr": S4_BACKBONE_LR},
        {"params": S4_HEAD_PARAMS, "lr": S4_HEAD_LR},
    ],
    weight_decay=S4_WEIGHT_DECAY,
)

S4_SCALER = torch.amp.GradScaler(
    "cuda",
    enabled=(S4_DEVICE.type == "cuda"),
)

def s4_to_device(batch):
    return {
        key: batch[key].to(S4_DEVICE, non_blocking=True)
        for key in (
            "input_ids", "char_ids", "char_to_token",
            "start_targets", "end_targets", "loss_mask",
        )
    }

def s4_boundary_loss(model, batch):
    start_logits, end_logits = model(
        batch["input_ids"],
        batch["char_ids"],
        batch["char_to_token"],
    )

    start_loss = F.binary_cross_entropy_with_logits(
        start_logits.float(),
        batch["start_targets"].float(),
        pos_weight=S4_POS_WEIGHTS_TENSOR,
        reduction="none",
    )
    end_loss = F.binary_cross_entropy_with_logits(
        end_logits.float(),
        batch["end_targets"].float(),
        pos_weight=S4_POS_WEIGHTS_TENSOR,
        reduction="none",
    )

    mask = batch["loss_mask"].float()
    denom = mask.sum().clamp_min(1.0)

    loss = (
        (start_loss * mask).sum()
        + (end_loss * mask).sum()
    ) / (2.0 * denom)

    return loss, int(mask.sum().item())

@torch.no_grad()
def s4_validate_teacher():
    S4_TEACHER_MODEL.eval()
    total_loss, total_positions = 0.0, 0

    for raw_batch in S4_VAL_LOADER:
        batch = s4_to_device(raw_batch)
        with torch.autocast(
            device_type=S4_DEVICE.type,
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda"),
        ):
            loss, positions = s4_boundary_loss(
                S4_TEACHER_MODEL, batch
            )

        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite teacher validation loss.")

        total_loss += float(loss.item()) * max(positions, 1)
        total_positions += max(positions, 1)

    return total_loss / max(total_positions, 1)

S4_TEACHER_HISTORY = []
S4_BEST_VAL_LOSS = float("inf")
S4_TRAIN_START = time.time()

print("=" * 72)
print("S4 — SUPERVISED ARCHITECTURE D TEACHER")
print("=" * 72)
print(f"Total parameters:     {S4_TOTAL_COUNT:,}")
print(f"Trainable parameters: {S4_TRAINABLE_COUNT:,}")
print(f"Gradient accumulation: {S4_GRAD_ACCUM}")
print(f"Epochs: {S4_TEACHER_EPOCHS}")

for epoch in range(1, S4_TEACHER_EPOCHS + 1):
    S4_TEACHER_MODEL.train()
    S4_OPTIMIZER.zero_grad(set_to_none=True)

    epoch_loss_sum = 0.0
    epoch_batches = 0
    update_count = 0

    for batch_idx, raw_batch in enumerate(S4_TRAIN_LOADER, start=1):
        batch = s4_to_device(raw_batch)

        with torch.autocast(
            device_type=S4_DEVICE.type,
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda"),
        ):
            loss, _ = s4_boundary_loss(S4_TEACHER_MODEL, batch)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite training loss at epoch {epoch}, "
                f"batch {batch_idx}."
            )

        S4_SCALER.scale(loss / S4_GRAD_ACCUM).backward()
        epoch_loss_sum += float(loss.detach().item())
        epoch_batches += 1

        is_accumulation_step = batch_idx % S4_GRAD_ACCUM == 0
        is_last_batch = batch_idx == len(S4_TRAIN_LOADER)

        if is_accumulation_step or is_last_batch:
            S4_SCALER.unscale_(S4_OPTIMIZER)
            torch.nn.utils.clip_grad_norm_(
                [
                    p for p in S4_TEACHER_MODEL.parameters()
                    if p.requires_grad
                ],
                S4_MAX_GRAD_NORM,
            )
            S4_SCALER.step(S4_OPTIMIZER)
            S4_SCALER.update()
            S4_OPTIMIZER.zero_grad(set_to_none=True)
            update_count += 1

        if batch_idx % 150 == 0 or is_last_batch:
            print(
                f"Epoch {epoch}/{S4_TEACHER_EPOCHS} | "
                f"batch {batch_idx}/{len(S4_TRAIN_LOADER)} | "
                f"loss {epoch_loss_sum / epoch_batches:.4f}"
            )

    S4_TRAIN_LOSS = epoch_loss_sum / max(epoch_batches, 1)
    S4_VAL_LOSS = s4_validate_teacher()

    S4_RECORD = {
        "epoch": epoch,
        "train_loss": S4_TRAIN_LOSS,
        "validation_loss": S4_VAL_LOSS,
        "optimizer_updates": update_count,
        "elapsed_seconds": time.time() - S4_TRAIN_START,
    }
    S4_TEACHER_HISTORY.append(S4_RECORD)

    if S4_VAL_LOSS < S4_BEST_VAL_LOSS:
        S4_BEST_VAL_LOSS = S4_VAL_LOSS
        torch.save(
            {
                "model_state_dict": S4_TEACHER_MODEL.state_dict(),
                "epoch": epoch,
                "validation_loss": S4_VAL_LOSS,
                "labels": S4_LABELS,
                "char_to_id": S4_CHAR_TO_ID,
                "config": {
                    "architecture": "Architecture_D_TCN_Language_Model",
                    "fine_tuned_blocks": [6, 7, 8, 9],
                    "total_parameters": S4_TOTAL_COUNT,
                    "trainable_parameters": S4_TRAINABLE_COUNT,
                    "window_tokens": S4_WINDOW_TOKENS,
                    "window_stride": S4_WINDOW_STRIDE,
                },
            },
            S4_TEACHER_CKPT_PATH,
        )
        S4_BEST_EPOCH = epoch

    with open(S4_TEACHER_HISTORY_PATH, "w", encoding="utf-8") as f:
        json.dump(S4_TEACHER_HISTORY, f, indent=2)

    print(
        f"Epoch {epoch} complete | train loss={S4_TRAIN_LOSS:.4f} | "
        f"val loss={S4_VAL_LOSS:.4f} | best val={S4_BEST_VAL_LOSS:.4f}"
    )

# Restore best validation-loss checkpoint.
S4_BEST_TEACHER_CHECKPOINT = torch.load(
    S4_TEACHER_CKPT_PATH,
    map_location=S4_DEVICE,
    weights_only=False,
)
S4_TEACHER_MODEL.load_state_dict(
    S4_BEST_TEACHER_CHECKPOINT["model_state_dict"],
    strict=True,
)
S4_TEACHER_MODEL.eval()

S4_TEACHER_CONFIG = {
    "experiment": "S4 supervised Architecture D teacher",
    "best_epoch": int(S4_BEST_TEACHER_CHECKPOINT["epoch"]),
    "best_validation_loss": float(S4_BEST_TEACHER_CHECKPOINT["validation_loss"]),
    "training_history": S4_TEACHER_HISTORY,
    "train_documents": len(S4_TRAIN_DOC_IDS),
    "validation_documents": len(S4_VAL_DOC_IDS),
    "trainable_parameters": S4_TRAINABLE_COUNT,
    "total_parameters": S4_TOTAL_COUNT,
    "checkpoint": str(S4_TEACHER_CKPT_PATH),
    "fold3_gold_labels_used": False,
}

with open(S4_TEACHER_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(S4_TEACHER_CONFIG, f, indent=2)

print("\nTeacher training complete.")
print("Best epoch:", S4_BEST_TEACHER_CHECKPOINT["epoch"])
print("Best validation loss:", f"{S4_BEST_VAL_LOSS:.4f}")
print("Checkpoint:", S4_TEACHER_CKPT_PATH)
print("History:", S4_TEACHER_HISTORY_PATH)
print("Fold 3 gold labels used: NO")

S4 — SUPERVISED ARCHITECTURE D TEACHER
Total parameters:     114,831,608
Trainable parameters: 25,858,296
Gradient accumulation: 3
Epochs: 3
Epoch 1/3 | batch 150/903 | loss 0.0729
Epoch 1/3 | batch 300/903 | loss 0.0431
Epoch 1/3 | batch 450/903 | loss 0.0323
Epoch 1/3 | batch 600/903 | loss 0.0262
Epoch 1/3 | batch 750/903 | loss 0.0222
Epoch 1/3 | batch 900/903 | loss 0.0194
Epoch 1/3 | batch 903/903 | loss 0.0193
Epoch 1 complete | train loss=0.0193 | val loss=0.0045 | best val=0.0045
Epoch 2/3 | batch 150/903 | loss 0.0042
Epoch 2/3 | batch 300/903 | loss 0.0044
Epoch 2/3 | batch 450/903 | loss 0.0040
Epoch 2/3 | batch 600/903 | loss 0.0038
Epoch 2/3 | batch 750/903 | loss 0.0037
Epoch 2/3 | batch 900/903 | loss 0.0036
Epoch 2/3 | batch 903/903 | loss 0.0036
Epoch 2 complete | train loss=0.0036 | val loss=0.0032 | best val=0.0032
Epoch 3/3 | batch 150/903 | loss 0.0026
Epoch 3/3 | batch 300/903 | loss 0.0026
Epoch 3/3 | batch 450/903 | loss 0.0026
Epoch 3/3 | batch 600/903 | loss 

In [7]:
# ============================================================
# CELL 6 — Evaluate exact character-span F1 and calibrate threshold
# ============================================================

import numpy as np
import torch

S4_TEACHER_MODEL.eval()
S4_ID_TO_LABEL = {v: k for k, v in S4_LABEL_TO_ID.items()}
S4_WINDOW_PREDICTIONS = []

def s4_meta_int(x):
    return int(x.item()) if torch.is_tensor(x) else int(x)

# Collect predictions from the held-out validation windows.
with torch.no_grad():
    for batch in S4_VAL_LOADER:
        b = s4_to_device(batch)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda")
        ):
            start_logits, end_logits = S4_TEACHER_MODEL(
                b["input_ids"], b["char_ids"], b["char_to_token"]
            )

        start_logits = start_logits.float().cpu().numpy()
        end_logits = end_logits.float().cpu().numpy()

        for i, doc_id in enumerate(batch["doc_ids"]):
            cs = s4_meta_int(batch["char_starts"][i])
            ce = s4_meta_int(batch["char_ends"][i])
            n = min(ce - cs, start_logits.shape[1], end_logits.shape[1])
            if n <= 0:
                continue
            S4_WINDOW_PREDICTIONS.append((
                str(doc_id), cs, ce,
                start_logits[i, :n].copy(),
                end_logits[i, :n].copy()
            ))

# Average logits over overlapping windows for each document character.
S4_DOC_ENDS = {}
for doc_id, cs, ce, _, _ in S4_WINDOW_PREDICTIONS:
    S4_DOC_ENDS[doc_id] = max(S4_DOC_ENDS.get(doc_id, 0), ce)

S4_AGG = {}
for doc_id, length in S4_DOC_ENDS.items():
    S4_AGG[doc_id] = {
        "start": np.zeros((length, S4_NUM_LABELS), dtype=np.float32),
        "end": np.zeros((length, S4_NUM_LABELS), dtype=np.float32),
        "count": np.zeros(length, dtype=np.float32)
    }

for doc_id, cs, ce, sl, el in S4_WINDOW_PREDICTIONS:
    n = min(len(sl), len(el), ce - cs)
    S4_AGG[doc_id]["start"][cs:cs+n] += sl[:n]
    S4_AGG[doc_id]["end"][cs:cs+n] += el[:n]
    S4_AGG[doc_id]["count"][cs:cs+n] += 1

for doc_id, d in S4_AGG.items():
    denom = np.maximum(d["count"][:, None], 1.0)
    d["start"] = 1.0 / (1.0 + np.exp(-np.clip(d["start"] / denom, -30, 30)))
    d["end"] = 1.0 / (1.0 + np.exp(-np.clip(d["end"] / denom, -30, 30)))

# Normalize the gold annotations to exact (document, start, end, label) spans.
def s4_unpack_gold(item):
    if isinstance(item, dict):
        start = item.get("start", item.get("start_char", item.get("start_character")))
        end = item.get("end", item.get("end_char", item.get("end_character")))
        label = item.get("label", item.get("entity", item.get("type")))
        return int(start), int(end), str(label)
    values = list(item)
    if len(values) < 3:
        raise ValueError(f"Unexpected gold span format: {item}")
    if isinstance(values[0], str):
        label, start, end = values[:3]
    else:
        start, end, label = values[:3]
    if isinstance(label, (int, np.integer)):
        label = S4_ID_TO_LABEL[int(label)]
    return int(start), int(end), str(label)

S4_VAL_DOC_SET = set(S4_AGG)
S4_GOLD_SPANS = set()

for raw_doc_id, annotations in S4_PARSED_GOLD.items():
    doc_id = str(raw_doc_id)
    if doc_id not in S4_VAL_DOC_SET:
        continue
    for item in annotations:
        start, end, label = s4_unpack_gold(item)
        if label in S4_LABEL_TO_ID and 0 <= start < end <= S4_DOC_ENDS[doc_id]:
            S4_GOLD_SPANS.add((doc_id, start, end, label))

# Decode each label by pairing each candidate start with the next unused end.
def s4_decode_spans(threshold):
    predicted = set()
    for doc_id, d in S4_AGG.items():
        start_probs, end_probs = d["start"], d["end"]
        for label, label_id in S4_LABEL_TO_ID.items():
            starts = np.flatnonzero(start_probs[:, label_id] >= threshold)
            ends = np.flatnonzero(end_probs[:, label_id] >= threshold)
            j = 0
            for start in starts:
                while j < len(ends) and ends[j] < start:
                    j += 1
                if j == len(ends):
                    break
                end = int(ends[j]) + 1  # end boundary is inclusive at character level
                j += 1
                if end > int(start):
                    predicted.add((doc_id, int(start), end, label))
    return predicted

S4_EVALUATED_LABELS = sorted({x[3] for x in S4_GOLD_SPANS})

def s4_score(predicted):
    tp = len(predicted & S4_GOLD_SPANS)
    precision = tp / len(predicted) if predicted else 0.0
    recall = tp / len(S4_GOLD_SPANS) if S4_GOLD_SPANS else 0.0
    micro_f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    label_f1s = []
    for label in S4_EVALUATED_LABELS:
        p = {x for x in predicted if x[3] == label}
        g = {x for x in S4_GOLD_SPANS if x[3] == label}
        t = len(p & g)
        pr = t / len(p) if p else 0.0
        re = t / len(g) if g else 0.0
        label_f1s.append(2 * pr * re / (pr + re) if pr + re else 0.0)

    macro_f1 = float(np.mean(label_f1s)) if label_f1s else 0.0
    return macro_f1, micro_f1, precision, recall, len(predicted)

S4_THRESHOLD_RESULTS = []
for threshold in np.arange(0.10, 0.651, 0.05):
    pred = s4_decode_spans(float(threshold))
    macro, micro, precision, recall, n_pred = s4_score(pred)
    S4_THRESHOLD_RESULTS.append(
        (macro, micro, precision, recall, float(threshold), n_pred)
    )

S4_THRESHOLD_RESULTS.sort(reverse=True)
S4_BEST_MACRO, S4_BEST_MICRO, S4_BEST_PRECISION, S4_BEST_RECALL, S4_BEST_THRESHOLD, S4_BEST_NPRED = S4_THRESHOLD_RESULTS[0]

print("=" * 66)
print("S4 TEACHER — EXACT SPAN VALIDATION")
print("=" * 66)
print(f"Validation documents:       {len(S4_AGG)}")
print(f"Gold spans:                 {len(S4_GOLD_SPANS)}")
print(f"Labels with validation gold:{len(S4_EVALUATED_LABELS):>3}")
print(f"Best threshold:             {S4_BEST_THRESHOLD:.2f}")
print(f"Validation Macro-F1:        {S4_BEST_MACRO:.4f}")
print(f"Validation Micro-F1:        {S4_BEST_MICRO:.4f}")
print(f"Precision / Recall:         {S4_BEST_PRECISION:.4f} / {S4_BEST_RECALL:.4f}")
print(f"Predicted spans:            {S4_BEST_NPRED}")
print("\nTop threshold candidates:")
for row in S4_THRESHOLD_RESULTS[:5]:
    print(f"threshold={row[4]:.2f} | macro-F1={row[0]:.4f} | "
          f"micro-F1={row[1]:.4f} | P={row[2]:.3f} | R={row[3]:.3f} "
          f"| predicted={row[5]}")

S4 TEACHER — EXACT SPAN VALIDATION
Validation documents:       14
Gold spans:                 1861
Labels with validation gold: 12
Best threshold:             0.55
Validation Macro-F1:        0.1747
Validation Micro-F1:        0.1853
Precision / Recall:         0.1511 / 0.2397
Predicted spans:            2952

Top threshold candidates:
threshold=0.55 | macro-F1=0.1747 | micro-F1=0.1853 | P=0.151 | R=0.240 | predicted=2952
threshold=0.50 | macro-F1=0.1686 | micro-F1=0.1715 | P=0.134 | R=0.239 | predicted=3318
threshold=0.60 | macro-F1=0.1628 | micro-F1=0.1813 | P=0.156 | R=0.217 | predicted=2595
threshold=0.65 | macro-F1=0.1601 | micro-F1=0.1921 | P=0.173 | R=0.216 | predicted=2324
threshold=0.45 | macro-F1=0.1586 | micro-F1=0.1551 | P=0.116 | R=0.232 | predicted=3710


In [8]:
# ============================================================
# CELL 7 — Tune per-label thresholds on validation spans
# ============================================================

def s4_decode_one_label(label, threshold):
    label_id = S4_LABEL_TO_ID[label]
    predicted = set()

    for doc_id, d in S4_AGG.items():
        starts = np.flatnonzero(d["start"][:, label_id] >= threshold)
        ends = np.flatnonzero(d["end"][:, label_id] >= threshold)
        j = 0

        for start in starts:
            while j < len(ends) and ends[j] < start:
                j += 1
            if j >= len(ends):
                break

            end = int(ends[j]) + 1
            j += 1
            if end > int(start):
                predicted.add((doc_id, int(start), end, label))

    return predicted


S4_CALIBRATION_THRESHOLDS = np.arange(0.25, 0.851, 0.05)
S4_LABEL_THRESHOLDS = {}
S4_LABEL_PREDICTIONS = set()
S4_LABEL_CALIBRATION = []

for label in S4_EVALUATED_LABELS:
    gold_label = {x for x in S4_GOLD_SPANS if x[3] == label}
    best = None

    for threshold in S4_CALIBRATION_THRESHOLDS:
        pred_label = s4_decode_one_label(label, float(threshold))
        tp = len(pred_label & gold_label)
        precision = tp / len(pred_label) if pred_label else 0.0
        recall = tp / len(gold_label) if gold_label else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        # Prefer a higher threshold when F1 ties.
        candidate = (f1, float(threshold), pred_label, precision, recall)
        if best is None or candidate[0:2] > best[0:2]:
            best = candidate

    f1, threshold, pred_label, precision, recall = best
    S4_LABEL_THRESHOLDS[label] = threshold
    S4_LABEL_PREDICTIONS.update(pred_label)
    S4_LABEL_CALIBRATION.append((label, threshold, f1, precision, recall, len(pred_label)))

S4_CAL_MACRO, S4_CAL_MICRO, S4_CAL_PREC, S4_CAL_REC, S4_CAL_N = s4_score(
    S4_LABEL_PREDICTIONS
)

print("=" * 66)
print("S4 TEACHER — GLOBAL VS LABEL-SPECIFIC THRESHOLDS")
print("=" * 66)
print(f"Global threshold Macro-F1:  {S4_BEST_MACRO:.4f}")
print(f"Per-label threshold Macro-F1:{S4_CAL_MACRO:.4f}")
print(f"Per-label threshold Micro-F1:{S4_CAL_MICRO:.4f}")
print(f"Precision / Recall:         {S4_CAL_PREC:.4f} / {S4_CAL_REC:.4f}")
print(f"Predicted spans:            {S4_CAL_N}")
print("\nLabel thresholds (validation-tuned):")
for label, threshold, f1, precision, recall, n_pred in S4_LABEL_CALIBRATION:
    print(f"{label:12s} threshold={threshold:.2f} | F1={f1:.3f} "
          f"| P={precision:.3f} | R={recall:.3f} | predicted={n_pred}")

S4 TEACHER — GLOBAL VS LABEL-SPECIFIC THRESHOLDS
Global threshold Macro-F1:  0.1747
Per-label threshold Macro-F1:0.2113
Per-label threshold Micro-F1:0.2281
Precision / Recall:         0.2055 / 0.2563
Predicted spans:            2321

Label thresholds (validation-tuned):
A.COUNSEL    threshold=0.70 | F1=0.200 | P=0.191 | R=0.209 | predicted=47
APP          threshold=0.50 | F1=0.051 | P=0.039 | R=0.072 | predicted=179
AUTH         threshold=0.55 | F1=0.025 | P=0.018 | R=0.039 | predicted=501
CASENO       threshold=0.75 | F1=0.188 | P=0.243 | R=0.153 | predicted=148
COURT        threshold=0.65 | F1=0.361 | P=0.304 | R=0.443 | predicted=355
DATE         threshold=0.55 | F1=0.477 | P=0.452 | R=0.504 | predicted=259
JUDGE        threshold=0.50 | F1=0.226 | P=0.179 | R=0.304 | predicted=78
PREC         threshold=0.80 | F1=0.100 | P=0.115 | R=0.089 | predicted=209
R.COUNSEL    threshold=0.35 | F1=0.391 | P=0.342 | R=0.458 | predicted=79
RESP         threshold=0.60 | F1=0.100 | P=0.156 | R=0.07

In [9]:
# ============================================================
# CELL 8 — Initialize compact student and cache KD targets
# ============================================================

import copy
import gc
import torch

S4_KD_TEMPERATURE = 2.0
S4_STUDENT_BLOCKS = 6

# The first six backbone blocks were frozen during teacher training.
# Copy them and use the same teacher-trained NER head as initialization.
S4_COMPACT_BACKBONE = copy.deepcopy(S4_D_MODEL)
S4_COMPACT_BACKBONE.tcn_blocks = torch.nn.ModuleList(
    list(S4_COMPACT_BACKBONE.tcn_blocks[:S4_STUDENT_BLOCKS])
)

S4_STUDENT_MODEL = copy.deepcopy(S4_TEACHER_MODEL)
S4_STUDENT_MODEL.backbone = S4_COMPACT_BACKBONE

# Unfreeze student parameters, then freeze the large pretrained embedding.
for p in S4_STUDENT_MODEL.parameters():
    p.requires_grad = True
S4_STUDENT_MODEL.backbone.token_embedding.weight.requires_grad = False

S4_STUDENT_MODEL = S4_STUDENT_MODEL.to(S4_DEVICE)
S4_STUDENT_MODEL.train()
S4_TEACHER_MODEL.eval()

S4_STUDENT_TOTAL_PARAMS = sum(p.numel() for p in S4_STUDENT_MODEL.parameters())
S4_STUDENT_TRAINABLE_PARAMS = sum(
    p.numel() for p in S4_STUDENT_MODEL.parameters() if p.requires_grad
)

print("=" * 66)
print("S4 — COMPACT STUDENT INITIALIZED")
print("=" * 66)
print(f"Teacher parameters:          {sum(p.numel() for p in S4_TEACHER_MODEL.parameters()):,}")
print(f"Student parameters:          {S4_STUDENT_TOTAL_PARAMS:,}")
print(f"Student trainable parameters:{S4_STUDENT_TRAINABLE_PARAMS:>14,}")
print(f"TCN blocks:                  {len(S4_STUDENT_MODEL.backbone.tcn_blocks)}")

# Cache teacher logits on training windows for knowledge distillation.
# Key = document ID and character range, matching the window metadata.
S4_TRAIN_KD_CACHE = {}

with torch.no_grad():
    for batch_idx, batch in enumerate(S4_TRAIN_LOADER, start=1):
        b = s4_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda")
        ):
            teacher_start, teacher_end = S4_TEACHER_MODEL(
                b["input_ids"], b["char_ids"], b["char_to_token"]
            )

        teacher_start = teacher_start.float().cpu()
        teacher_end = teacher_end.float().cpu()

        for i, doc_id in enumerate(batch["doc_ids"]):
            cs = s4_meta_int(batch["char_starts"][i])
            ce = s4_meta_int(batch["char_ends"][i])
            n = min(ce - cs, teacher_start.shape[1], teacher_end.shape[1])
            if n <= 0:
                continue

            key = (str(doc_id), cs, ce)
            if key in S4_TRAIN_KD_CACHE:
                raise ValueError(f"Duplicate training-window cache key: {key}")

            S4_TRAIN_KD_CACHE[key] = (
                teacher_start[i, :n].half().contiguous(),
                teacher_end[i, :n].half().contiguous()
            )

        if batch_idx % 200 == 0 or batch_idx == len(S4_TRAIN_LOADER):
            print(f"Cached teacher targets: {batch_idx}/{len(S4_TRAIN_LOADER)}")

gc.collect()
if S4_DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"\nTraining windows cached: {len(S4_TRAIN_KD_CACHE)}")
print("Student initialization and teacher-target caching complete.")

S4 — COMPACT STUDENT INITIALIZED
Teacher parameters:          114,831,608
Student parameters:          89,649,400
Student trainable parameters:    38,449,400
TCN blocks:                  6
Cached teacher targets: 200/903
Cached teacher targets: 400/903
Cached teacher targets: 600/903
Cached teacher targets: 800/903
Cached teacher targets: 903/903

Training windows cached: 903
Student initialization and teacher-target caching complete.


In [10]:
# ============================================================
# CELL 9 — Train compact student using supervised + KD losses
# ============================================================

import os
import gc
import json
import torch
import torch.nn.functional as F

S4_STUDENT_EPOCHS = 3
S4_STUDENT_ACCUMULATION = 3
S4_KD_WEIGHT = 0.20
S4_STUDENT_DIR = "/kaggle/working/S4_Architecture_D_KD"
os.makedirs(S4_STUDENT_DIR, exist_ok=True)

S4_BACKBONE_PARAMS = [
    p for n, p in S4_STUDENT_MODEL.named_parameters()
    if n.startswith("backbone.") and p.requires_grad
]
S4_HEAD_PARAMS = [
    p for n, p in S4_STUDENT_MODEL.named_parameters()
    if not n.startswith("backbone.") and p.requires_grad
]

S4_STUDENT_OPTIMIZER = torch.optim.AdamW(
    [
        {"params": S4_BACKBONE_PARAMS, "lr": 1e-5},
        {"params": S4_HEAD_PARAMS, "lr": 2e-4},
    ],
    weight_decay=0.01
)
S4_STUDENT_SCALER = torch.amp.GradScaler(
    "cuda", enabled=(S4_DEVICE.type == "cuda")
)

# ============================================================
# CELL 9 PATCH — Replace s4_student_losses() with this function
# ============================================================

def s4_student_losses(batch_device, batch_original, calculate_kd=True):
    start_logits, end_logits = S4_STUDENT_MODEL(
        batch_device["input_ids"],
        batch_device["char_ids"],
        batch_device["char_to_token"]
    )
    start_logits = start_logits.float()
    end_logits = end_logits.float()

    start_target = batch_device["start_targets"].float()
    end_target = batch_device["end_targets"].float()

    batch_size, seq_len, num_labels = start_logits.shape

    # Normalize loss_mask to [batch, characters].
    mask = batch_device["loss_mask"].float()

    if mask.ndim >= 2 and mask.shape[0] != batch_size:
        if mask.shape[1] == batch_size:
            mask = mask.transpose(0, 1)

    while mask.ndim > 2 and 1 in mask.shape[1:]:
        singleton_dim = next(
            i for i in range(1, mask.ndim) if mask.shape[i] == 1
        )
        mask = mask.squeeze(singleton_dim)

    if mask.ndim > 2:
        if mask.shape[1] == seq_len:
            while mask.ndim > 2:
                mask = mask.amax(dim=-1)
        elif mask.shape[-1] == seq_len:
            while mask.ndim > 2:
                mask = mask.amax(dim=1)

    if mask.ndim == 1 and batch_size == 1 and mask.numel() == seq_len:
        mask = mask.unsqueeze(0)

    if mask.shape != (batch_size, seq_len):
        raise RuntimeError(
            f"Could not normalize loss_mask: mask={tuple(mask.shape)}, "
            f"logits={tuple(start_logits.shape)}, "
            f"original mask={tuple(batch_device['loss_mask'].shape)}"
        )

    if start_target.shape != start_logits.shape or end_target.shape != end_logits.shape:
        raise RuntimeError(
            f"Target/logit mismatch: start target={tuple(start_target.shape)}, "
            f"start logits={tuple(start_logits.shape)}, "
            f"end target={tuple(end_target.shape)}, "
            f"end logits={tuple(end_logits.shape)}"
        )

    mask3 = mask.unsqueeze(-1)
    pos_weights = S4_POS_WEIGHTS_TENSOR.to(S4_DEVICE).float()

    start_bce = F.binary_cross_entropy_with_logits(
        start_logits, start_target,
        pos_weight=pos_weights, reduction="none"
    )
    end_bce = F.binary_cross_entropy_with_logits(
        end_logits, end_target,
        pos_weight=pos_weights, reduction="none"
    )

    denom = (mask.sum() * num_labels * 2).clamp_min(1.0)
    supervised_loss = ((start_bce + end_bce) * mask3).sum() / denom

    if not calculate_kd:
        return supervised_loss, None

    kd_terms = []
    temperature = S4_KD_TEMPERATURE

    for i, doc_id in enumerate(batch_original["doc_ids"]):
        cs = s4_meta_int(batch_original["char_starts"][i])
        ce = s4_meta_int(batch_original["char_ends"][i])
        key = (str(doc_id), cs, ce)

        if key not in S4_TRAIN_KD_CACHE:
            raise KeyError(f"Missing teacher target for window: {key}")

        teacher_start, teacher_end = S4_TRAIN_KD_CACHE[key]
        n = min(
            teacher_start.shape[0],
            teacher_end.shape[0],
            start_logits.shape[1],
            end_logits.shape[1],
            ce - cs
        )
        if n <= 0:
            continue

        ts = teacher_start[:n].to(S4_DEVICE, dtype=torch.float32) / temperature
        te = teacher_end[:n].to(S4_DEVICE, dtype=torch.float32) / temperature
        ps = torch.sigmoid(ts).clamp(1e-6, 1 - 1e-6)
        pe = torch.sigmoid(te).clamp(1e-6, 1 - 1e-6)

        ss = start_logits[i, :n] / temperature
        se = end_logits[i, :n] / temperature

        def bernoulli_kl(p, logits):
            return (
                p * (torch.log(p) - F.logsigmoid(logits))
                + (1 - p) * (
                    torch.log1p(-p) - F.logsigmoid(-logits)
                )
            )

        m = mask[i, :n].unsqueeze(-1)
        kd_denom = (m.sum() * num_labels * 2).clamp_min(1.0)

        kd_loss = (
            (bernoulli_kl(ps, ss) * m).sum()
            + (bernoulli_kl(pe, se) * m).sum()
        ) / kd_denom

        kd_terms.append(kd_loss * (temperature ** 2))

    kd_loss = (
        torch.stack(kd_terms).mean()
        if kd_terms else supervised_loss.new_zeros(())
    )

    return supervised_loss, kd_loss
@torch.no_grad()
def s4_validate_student():
    S4_STUDENT_MODEL.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in S4_VAL_LOADER:
        b = s4_to_device(batch)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda")
        ):
            loss, _ = s4_student_losses(b, batch, calculate_kd=False)

        if torch.isfinite(loss):
            total_loss += float(loss.item())
            total_batches += 1

    S4_STUDENT_MODEL.train()
    return total_loss / max(total_batches, 1)


S4_STUDENT_HISTORY = []
S4_STUDENT_BEST_VAL = float("inf")
S4_STUDENT_BEST_EPOCH = 0
S4_STUDENT_CKPT_PATH = os.path.join(S4_STUDENT_DIR, "S4_student_best.pt")

print("=" * 66)
print("S4 — COMPACT STUDENT TRAINING")
print("=" * 66)
print(f"Epochs: {S4_STUDENT_EPOCHS}")
print(f"Gradient accumulation: {S4_STUDENT_ACCUMULATION}")
print(f"KD weight: {S4_KD_WEIGHT:.2f}")
print(f"Backbone LR: 1e-5 | NER-head LR: 2e-4")
print("-" * 66)

for epoch in range(1, S4_STUDENT_EPOCHS + 1):
    S4_STUDENT_MODEL.train()
    S4_STUDENT_OPTIMIZER.zero_grad(set_to_none=True)
    epoch_total = 0.0
    epoch_sup = 0.0
    epoch_kd = 0.0
    updates = 0

    for batch_idx, batch in enumerate(S4_TRAIN_LOADER, start=1):
        b = s4_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda")
        ):
            sup_loss, kd_loss = s4_student_losses(b, batch, calculate_kd=True)
            total_loss = (1.0 - S4_KD_WEIGHT) * sup_loss + S4_KD_WEIGHT * kd_loss

        if not torch.isfinite(total_loss):
            S4_STUDENT_OPTIMIZER.zero_grad(set_to_none=True)
            raise RuntimeError(
                f"Non-finite student loss at epoch {epoch}, batch {batch_idx}"
            )

        S4_STUDENT_SCALER.scale(
            total_loss / S4_STUDENT_ACCUMULATION
        ).backward()

        epoch_total += float(total_loss.detach().item())
        epoch_sup += float(sup_loss.detach().item())
        epoch_kd += float(kd_loss.detach().item())

        if (
            batch_idx % S4_STUDENT_ACCUMULATION == 0
            or batch_idx == len(S4_TRAIN_LOADER)
        ):
            S4_STUDENT_SCALER.unscale_(S4_STUDENT_OPTIMIZER)
            torch.nn.utils.clip_grad_norm_(
                [p for p in S4_STUDENT_MODEL.parameters() if p.requires_grad],
                max_norm=1.0
            )
            S4_STUDENT_SCALER.step(S4_STUDENT_OPTIMIZER)
            S4_STUDENT_SCALER.update()
            S4_STUDENT_OPTIMIZER.zero_grad(set_to_none=True)
            updates += 1

        if batch_idx % 200 == 0 or batch_idx == len(S4_TRAIN_LOADER):
            print(
                f"Epoch {epoch}/{S4_STUDENT_EPOCHS} | "
                f"batch {batch_idx}/{len(S4_TRAIN_LOADER)} | "
                f"loss={epoch_total/batch_idx:.4f} | "
                f"supervised={epoch_sup/batch_idx:.4f} | "
                f"KD={epoch_kd/batch_idx:.4f}"
            )

    val_loss = s4_validate_student()
    record = {
        "epoch": epoch,
        "train_loss": epoch_total / len(S4_TRAIN_LOADER),
        "train_supervised_loss": epoch_sup / len(S4_TRAIN_LOADER),
        "train_kd_loss": epoch_kd / len(S4_TRAIN_LOADER),
        "validation_loss": val_loss
    }
    S4_STUDENT_HISTORY.append(record)

    print(
        f"Epoch {epoch} complete | "
        f"train={record['train_loss']:.4f} | "
        f"val={val_loss:.4f}"
    )

    if val_loss < S4_STUDENT_BEST_VAL:
        S4_STUDENT_BEST_VAL = val_loss
        S4_STUDENT_BEST_EPOCH = epoch
        torch.save(
            {
                "model_state_dict": {
                    k: v.detach().cpu()
                    for k, v in S4_STUDENT_MODEL.state_dict().items()
                },
                "epoch": epoch,
                "validation_loss": val_loss,
                "labels": S4_LABEL_TO_ID,
                "char_to_id": S4_CHAR_TO_ID,
                "num_blocks": S4_STUDENT_BLOCKS,
                "temperature": S4_KD_TEMPERATURE,
                "kd_weight": S4_KD_WEIGHT,
                "label_thresholds": S4_LABEL_THRESHOLDS
            },
            S4_STUDENT_CKPT_PATH
        )
        print("  Saved new best student checkpoint.")

with open(os.path.join(S4_STUDENT_DIR, "S4_student_history.json"), "w") as f:
    json.dump(S4_STUDENT_HISTORY, f, indent=2)

# Restore the best validation-loss checkpoint.
S4_STUDENT_CHECKPOINT = torch.load(
    S4_STUDENT_CKPT_PATH, map_location=S4_DEVICE, weights_only=False
)
S4_STUDENT_MODEL.load_state_dict(S4_STUDENT_CHECKPOINT["model_state_dict"])
S4_STUDENT_MODEL.eval()

gc.collect()
if S4_DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nStudent training complete.")
print(f"Best epoch: {S4_STUDENT_BEST_EPOCH}")
print(f"Best validation loss: {S4_STUDENT_BEST_VAL:.4f}")
print(f"Checkpoint: {S4_STUDENT_CKPT_PATH}")

S4 — COMPACT STUDENT TRAINING
Epochs: 3
Gradient accumulation: 3
KD weight: 0.20
Backbone LR: 1e-5 | NER-head LR: 2e-4
------------------------------------------------------------------
Epoch 1/3 | batch 200/903 | loss=0.0027 | supervised=0.0023 | KD=0.0043
Epoch 1/3 | batch 400/903 | loss=0.0026 | supervised=0.0023 | KD=0.0039
Epoch 1/3 | batch 600/903 | loss=0.0025 | supervised=0.0022 | KD=0.0037
Epoch 1/3 | batch 800/903 | loss=0.0024 | supervised=0.0021 | KD=0.0036
Epoch 1/3 | batch 903/903 | loss=0.0024 | supervised=0.0021 | KD=0.0035
Epoch 1 complete | train=0.0024 | val=0.0028
  Saved new best student checkpoint.
Epoch 2/3 | batch 200/903 | loss=0.0019 | supervised=0.0017 | KD=0.0029
Epoch 2/3 | batch 400/903 | loss=0.0020 | supervised=0.0018 | KD=0.0030
Epoch 2/3 | batch 600/903 | loss=0.0020 | supervised=0.0018 | KD=0.0029
Epoch 2/3 | batch 800/903 | loss=0.0020 | supervised=0.0018 | KD=0.0029
Epoch 2/3 | batch 903/903 | loss=0.0020 | supervised=0.0018 | KD=0.0029
Epoch 2 comp

In [11]:
# ============================================================
# CELL 10 — Student exact-span evaluation and threshold tuning
# ============================================================

import numpy as np
import torch

S4_STUDENT_MODEL.eval()
S4_STUDENT_WINDOW_LOGITS = []

with torch.no_grad():
    for batch_idx, batch in enumerate(S4_VAL_LOADER, start=1):
        b = s4_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda")
        ):
            start_logits, end_logits = S4_STUDENT_MODEL(
                b["input_ids"], b["char_ids"], b["char_to_token"]
            )

        start_logits = start_logits.float().cpu().numpy()
        end_logits = end_logits.float().cpu().numpy()

        for i, doc_id in enumerate(batch["doc_ids"]):
            cs = s4_meta_int(batch["char_starts"][i])
            ce = s4_meta_int(batch["char_ends"][i])
            n = min(ce - cs, start_logits.shape[1], end_logits.shape[1])
            if n > 0:
                S4_STUDENT_WINDOW_LOGITS.append(
                    (str(doc_id), cs, ce, start_logits[i, :n].copy(),
                     end_logits[i, :n].copy())
                )

# Average logits where validation windows overlap.
S4_STUDENT_DOC_LENGTHS = {}
for doc_id, cs, ce, _, _ in S4_STUDENT_WINDOW_LOGITS:
    S4_STUDENT_DOC_LENGTHS[doc_id] = max(
        S4_STUDENT_DOC_LENGTHS.get(doc_id, 0), ce
    )

S4_STUDENT_AGG = {
    doc_id: {
        "start": np.zeros((length, S4_NUM_LABELS), dtype=np.float32),
        "end": np.zeros((length, S4_NUM_LABELS), dtype=np.float32),
        "count": np.zeros(length, dtype=np.float32)
    }
    for doc_id, length in S4_STUDENT_DOC_LENGTHS.items()
}

for doc_id, cs, ce, sl, el in S4_STUDENT_WINDOW_LOGITS:
    n = min(len(sl), len(el), ce - cs)
    d = S4_STUDENT_AGG[doc_id]
    d["start"][cs:cs+n] += sl[:n]
    d["end"][cs:cs+n] += el[:n]
    d["count"][cs:cs+n] += 1

for d in S4_STUDENT_AGG.values():
    denom = np.maximum(d["count"][:, None], 1.0)
    d["start"] = 1.0 / (
        1.0 + np.exp(-np.clip(d["start"] / denom, -30, 30))
    )
    d["end"] = 1.0 / (
        1.0 + np.exp(-np.clip(d["end"] / denom, -30, 30))
    )

def s4_student_decode(thresholds):
    # thresholds can be one scalar or a label -> threshold dictionary.
    pred = set()
    for doc_id, d in S4_STUDENT_AGG.items():
        for label, label_id in S4_LABEL_TO_ID.items():
            threshold = (
                float(thresholds[label])
                if isinstance(thresholds, dict)
                else float(thresholds)
            )
            starts = np.flatnonzero(d["start"][:, label_id] >= threshold)
            ends = np.flatnonzero(d["end"][:, label_id] >= threshold)
            j = 0

            for start in starts:
                while j < len(ends) and ends[j] < start:
                    j += 1
                if j >= len(ends):
                    break

                end = int(ends[j]) + 1
                j += 1
                if end > int(start):
                    pred.add((doc_id, int(start), end, label))
    return pred

# Global threshold sweep.
S4_STUDENT_GLOBAL_RESULTS = []
for threshold in np.arange(0.10, 0.851, 0.05):
    pred = s4_student_decode(float(threshold))
    macro, micro, precision, recall, count = s4_score(pred)
    S4_STUDENT_GLOBAL_RESULTS.append(
        (macro, micro, precision, recall, float(threshold), count)
    )

S4_STUDENT_GLOBAL_RESULTS.sort(reverse=True)
S4_STUDENT_GLOBAL_BEST = S4_STUDENT_GLOBAL_RESULTS[0]

# Tune one threshold per label on the same validation set.
S4_STUDENT_LABEL_THRESHOLDS = {}
S4_STUDENT_LABEL_PREDICTIONS = set()
S4_STUDENT_LABEL_RESULTS = []

for label in S4_EVALUATED_LABELS:
    label_id = S4_LABEL_TO_ID[label]
    gold = {x for x in S4_GOLD_SPANS if x[3] == label}
    best = None

    for threshold in np.arange(0.25, 0.851, 0.05):
        pred_label = set()

        for doc_id, d in S4_STUDENT_AGG.items():
            starts = np.flatnonzero(d["start"][:, label_id] >= threshold)
            ends = np.flatnonzero(d["end"][:, label_id] >= threshold)
            j = 0

            for start in starts:
                while j < len(ends) and ends[j] < start:
                    j += 1
                if j >= len(ends):
                    break
                end = int(ends[j]) + 1
                j += 1
                if end > int(start):
                    pred_label.add((doc_id, int(start), end, label))

        tp = len(pred_label & gold)
        precision = tp / len(pred_label) if pred_label else 0.0
        recall = tp / len(gold) if gold else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        candidate = (f1, float(threshold), pred_label, precision, recall)
        if best is None or candidate[:2] > best[:2]:
            best = candidate

    f1, threshold, pred_label, precision, recall = best
    S4_STUDENT_LABEL_THRESHOLDS[label] = threshold
    S4_STUDENT_LABEL_PREDICTIONS.update(pred_label)
    S4_STUDENT_LABEL_RESULTS.append(
        (label, threshold, f1, precision, recall, len(pred_label))
    )

S4_STUDENT_CAL_MACRO, S4_STUDENT_CAL_MICRO, S4_STUDENT_CAL_P, \
    S4_STUDENT_CAL_R, S4_STUDENT_CAL_N = s4_score(
        S4_STUDENT_LABEL_PREDICTIONS
    )

print("=" * 68)
print("S4 — STUDENT EXACT-SPAN VALIDATION")
print("=" * 68)
print(f"Validation documents:            {len(S4_STUDENT_AGG)}")
print(f"Gold spans:                      {len(S4_GOLD_SPANS)}")
print(f"Student best global threshold:   {S4_STUDENT_GLOBAL_BEST[4]:.2f}")
print(f"Student global Macro-F1:         {S4_STUDENT_GLOBAL_BEST[0]:.4f}")
print(f"Student global Micro-F1:         {S4_STUDENT_GLOBAL_BEST[1]:.4f}")
print(f"Student label-tuned Macro-F1:    {S4_STUDENT_CAL_MACRO:.4f}")
print(f"Student label-tuned Micro-F1:    {S4_STUDENT_CAL_MICRO:.4f}")
print(f"Label-tuned precision / recall:  {S4_STUDENT_CAL_P:.4f} / {S4_STUDENT_CAL_R:.4f}")
print(f"Label-tuned predicted spans:     {S4_STUDENT_CAL_N}")

print("\nTeacher comparison (same validation split):")
print(f"Teacher global Macro-F1:         {S4_BEST_MACRO:.4f}")
print(f"Teacher label-tuned Macro-F1:    {S4_CAL_MACRO:.4f}")
print(f"Student global Macro-F1:         {S4_STUDENT_GLOBAL_BEST[0]:.4f}")
print(f"Student label-tuned Macro-F1:    {S4_STUDENT_CAL_MACRO:.4f}")

print("\nStudent label thresholds:")
for label, threshold, f1, precision, recall, count in S4_STUDENT_LABEL_RESULTS:
    print(
        f"{label:12s} threshold={threshold:.2f} | F1={f1:.3f} "
        f"| P={precision:.3f} | R={recall:.3f} | predicted={count}"
    )

S4 — STUDENT EXACT-SPAN VALIDATION
Validation documents:            14
Gold spans:                      1861
Student best global threshold:   0.80
Student global Macro-F1:         0.1893
Student global Micro-F1:         0.2321
Student label-tuned Macro-F1:    0.2202
Student label-tuned Micro-F1:    0.2521
Label-tuned precision / recall:  0.2561 / 0.2483
Label-tuned predicted spans:     1804

Teacher comparison (same validation split):
Teacher global Macro-F1:         0.1747
Teacher label-tuned Macro-F1:    0.2113
Student global Macro-F1:         0.1893
Student label-tuned Macro-F1:    0.2202

Student label thresholds:
A.COUNSEL    threshold=0.70 | F1=0.211 | P=0.192 | R=0.233 | predicted=52
APP          threshold=0.75 | F1=0.066 | P=0.093 | R=0.052 | predicted=54
AUTH         threshold=0.80 | F1=0.021 | P=0.020 | R=0.022 | predicted=249
CASENO       threshold=0.80 | F1=0.310 | P=0.515 | R=0.221 | predicted=101
COURT        threshold=0.30 | F1=0.257 | P=0.192 | R=0.389 | predicted=496
D

In [12]:
# ============================================================
# PATCH — Restore the original training collator for inference
# ============================================================

if "S4_COLLATE_WINDOWS" not in globals():
    if "S4_TRAIN_LOADER" not in globals():
        raise NameError(
            "S4_TRAIN_LOADER is also missing. The notebook kernel may have reset."
        )
    S4_COLLATE_WINDOWS = S4_TRAIN_LOADER.collate_fn

print("Inference collator ready:", S4_COLLATE_WINDOWS)

Inference collator ready: <function s4_collate_windows at 0x78a08be48040>


In [13]:
# ============================================================
# CELL 11 — Load Fold 3 JSON, predict, validate and ZIP
# ============================================================

import json
import zipfile
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader

S4_TEST_JSON_PATH = Path(
    "/kaggle/working/phase2_lner/Phase_2_LNER/Test/fold3-text.json"
)
S4_OUTPUT_DIR = Path("/kaggle/working/S4_Architecture_D_KD")
S4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not S4_TEST_JSON_PATH.exists():
    raise FileNotFoundError(f"Fold 3 JSON not found: {S4_TEST_JSON_PATH}")

with open(S4_TEST_JSON_PATH, "r", encoding="utf-8") as f:
    S4_RAW_TEST_JSON = json.load(f)

# Convert supported JSON layouts to {document_id: document_text}.
S4_TEST_TEXTS = {}
S4_ID_FIELDS = ("doc_id", "document_id", "documentId", "id", "docid")
S4_TEXT_FIELDS = ("text", "content", "document_text", "raw_text", "body")
S4_WRAPPER_FIELDS = ("documents", "data", "texts", "docs", "test", "fold3")

def s4_add_test_record(doc_id, value):
    if isinstance(value, str):
        if doc_id is None:
            return False
        key, text = str(doc_id), value

    elif isinstance(value, dict):
        text_key = next(
            (k for k in S4_TEXT_FIELDS if isinstance(value.get(k), str)),
            None
        )
        if text_key is None:
            return False

        internal_id = next(
            (value[k] for k in S4_ID_FIELDS if value.get(k) is not None),
            None
        )
        key = str(internal_id if internal_id is not None else doc_id)
        text = value[text_key]

        if key == "None":
            return False
    else:
        return False

    if key in S4_TEST_TEXTS:
        raise ValueError(f"Duplicate test document ID: {key}")

    S4_TEST_TEXTS[key] = text
    return True

def s4_parse_test_json(obj):
    # A single document record.
    if isinstance(obj, dict):
        has_text = any(
            isinstance(obj.get(k), str) for k in S4_TEXT_FIELDS
        )
        has_id = any(obj.get(k) is not None for k in S4_ID_FIELDS)

        if has_text and has_id:
            if s4_add_test_record(None, obj):
                return

        # First try common wrapper fields.
        for key in S4_WRAPPER_FIELDS:
            if key in obj and isinstance(obj[key], (dict, list)):
                before = len(S4_TEST_TEXTS)
                s4_parse_test_json(obj[key])
                if len(S4_TEST_TEXTS) > before:
                    return

        # Otherwise treat it as a document-ID mapping or record mapping.
        for key, value in obj.items():
            if not s4_add_test_record(key, value):
                if isinstance(value, (dict, list)):
                    s4_parse_test_json(value)

    elif isinstance(obj, list):
        for record in obj:
            if not s4_add_test_record(None, record):
                if isinstance(record, (dict, list)):
                    s4_parse_test_json(record)

s4_parse_test_json(S4_RAW_TEST_JSON)

if not S4_TEST_TEXTS:
    if isinstance(S4_RAW_TEST_JSON, dict):
        print("Top-level JSON keys:", list(S4_RAW_TEST_JSON.keys())[:20])
        for k, v in list(S4_RAW_TEST_JSON.items())[:3]:
            print(f"Sample structure for {k!r}: type={type(v).__name__}")
            if isinstance(v, dict):
                print("  Fields:", list(v.keys())[:20])
            elif isinstance(v, list) and v:
                print("  First record type:", type(v[0]).__name__)
                if isinstance(v[0], dict):
                    print("  First record fields:", list(v[0].keys())[:20])
    raise ValueError("Could not parse Fold 3 JSON into document IDs and texts.")

print(f"Fold 3 documents loaded: {len(S4_TEST_TEXTS)}")
print("Document IDs and character lengths:")
for doc_id, doc_text in list(S4_TEST_TEXTS.items())[:5]:
    print(f"  {doc_id}: {len(doc_text):,} characters")

# Build windows using test text only; no Fold 3 gold labels are loaded.
S4_TEST_WINDOWS = []
for doc_id, doc_text in S4_TEST_TEXTS.items():
    windows, _, _ = s4_build_windows(doc_id, doc_text, [])
    S4_TEST_WINDOWS.extend(windows)

if not S4_TEST_WINDOWS:
    raise ValueError("No inference windows were produced for Fold 3.")

S4_TEST_LOADER = DataLoader(
    S4_TEST_WINDOWS,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=S4_COLLATE_WINDOWS
)

# Aggregate overlapping window logits at character level.
S4_TEST_AGG = {
    doc_id: {
        "start": np.zeros((len(txt), S4_NUM_LABELS), dtype=np.float32),
        "end": np.zeros((len(txt), S4_NUM_LABELS), dtype=np.float32),
        "count": np.zeros(len(txt), dtype=np.float32)
    }
    for doc_id, txt in S4_TEST_TEXTS.items()
}

S4_STUDENT_MODEL.eval()

with torch.no_grad():
    for batch_idx, batch in enumerate(S4_TEST_LOADER, start=1):
        b = s4_to_device(batch)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(S4_DEVICE.type == "cuda")
        ):
            start_logits, end_logits = S4_STUDENT_MODEL(
                b["input_ids"], b["char_ids"], b["char_to_token"]
            )

        start_logits = start_logits.float().cpu().numpy()
        end_logits = end_logits.float().cpu().numpy()

        for i, raw_id in enumerate(batch["doc_ids"]):
            doc_id = str(raw_id)
            cs = s4_meta_int(batch["char_starts"][i])
            ce = s4_meta_int(batch["char_ends"][i])
            doc_len = len(S4_TEST_TEXTS[doc_id])

            cs = max(0, cs)
            n = min(
                ce - cs, start_logits.shape[1],
                end_logits.shape[1], max(0, doc_len - cs)
            )
            if n <= 0:
                continue

            d = S4_TEST_AGG[doc_id]
            d["start"][cs:cs+n] += start_logits[i, :n]
            d["end"][cs:cs+n] += end_logits[i, :n]
            d["count"][cs:cs+n] += 1

        if batch_idx % 200 == 0 or batch_idx == len(S4_TEST_LOADER):
            print(f"Inference windows: {batch_idx}/{len(S4_TEST_LOADER)}")

for d in S4_TEST_AGG.values():
    denom = np.maximum(d["count"][:, None], 1.0)
    d["start"] = 1.0 / (
        1.0 + np.exp(-np.clip(d["start"] / denom, -30, 30))
    )
    d["end"] = 1.0 / (
        1.0 + np.exp(-np.clip(d["end"] / denom, -30, 30))
    )

# Decode with validation-calibrated per-label thresholds.
S4_PREDICTIONS = {doc_id: [] for doc_id in S4_TEST_TEXTS}

for doc_id, d in S4_TEST_AGG.items():
    doc_len = len(S4_TEST_TEXTS[doc_id])
    spans = set()

    for label, label_id in S4_LABEL_TO_ID.items():
        threshold = float(S4_STUDENT_LABEL_THRESHOLDS[label])
        starts = np.flatnonzero(d["start"][:, label_id] >= threshold)
        ends = np.flatnonzero(d["end"][:, label_id] >= threshold)

        j = 0
        for start in starts:
            while j < len(ends) and ends[j] < start:
                j += 1
            if j >= len(ends):
                break

            # Boundary end is inclusive; submission end is exclusive.
            end = int(ends[j]) + 1
            j += 1

            if 0 <= int(start) < end <= doc_len:
                spans.add((int(start), end, label))

    S4_PREDICTIONS[doc_id] = [
        [start, end, label]
        for start, end, label in sorted(spans)
    ]

# Validate prediction.json before packaging.
S4_PREDICTION_JSON = S4_OUTPUT_DIR / "prediction.json"
S4_SUBMISSION_ZIP = Path(
    "/kaggle/working/S4_Architecture_D_KD_Submission.zip"
)

S4_ERRORS = []
for doc_id, spans in S4_PREDICTIONS.items():
    doc_len = len(S4_TEST_TEXTS[doc_id])
    seen = set()

    for span in spans:
        if (
            len(span) != 3
            or not isinstance(span[0], int)
            or not isinstance(span[1], int)
            or not isinstance(span[2], str)
            or not (0 <= span[0] < span[1] <= doc_len)
            or span[2] not in S4_LABEL_TO_ID
            or tuple(span) in seen
        ):
            S4_ERRORS.append((doc_id, span))
        seen.add(tuple(span))

if S4_ERRORS:
    raise ValueError(f"Invalid predictions detected: {S4_ERRORS[:5]}")

with open(S4_PREDICTION_JSON, "w", encoding="utf-8") as f:
    json.dump(S4_PREDICTIONS, f, ensure_ascii=False)

with zipfile.ZipFile(S4_SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(S4_PREDICTION_JSON, arcname="prediction.json")

with zipfile.ZipFile(S4_SUBMISSION_ZIP, "r") as zf:
    assert zf.namelist() == ["prediction.json"]
    with zf.open("prediction.json") as f:
        S4_CHECK = json.load(f)

assert set(S4_CHECK) == set(S4_TEST_TEXTS)
assert all(isinstance(value, list) for value in S4_CHECK.values())

print("\n" + "=" * 66)
print("S4 FOLD 3 SUBMISSION — VERIFIED")
print("=" * 66)
print(f"Documents:           {len(S4_PREDICTIONS)}")
print(f"Predicted spans:     {sum(len(v) for v in S4_PREDICTIONS.values())}")
print(f"Documents nonempty:  {sum(bool(v) for v in S4_PREDICTIONS.values())}")
print(f"Documents empty:     {sum(not v for v in S4_PREDICTIONS.values())}")
print(f"Invalid spans:       {len(S4_ERRORS)}")
print("Fold 3 gold labels used: NO")
print(f"Submission ZIP:      {S4_SUBMISSION_ZIP}")
print("ZIP root contents:   prediction.json")

Fold 3 documents loaded: 35
Document IDs and character lengths:
  1128835: 51,797 characters
  450185: 27,736 characters
  13370078: 14,769 characters
  83838189: 27,021 characters
  1670685: 602 characters
Inference windows: 200/637
Inference windows: 400/637
Inference windows: 600/637
Inference windows: 637/637

S4 FOLD 3 SUBMISSION — VERIFIED
Documents:           35
Predicted spans:     4130
Documents nonempty:  34
Documents empty:     1
Invalid spans:       0
Fold 3 gold labels used: NO
Submission ZIP:      /kaggle/working/S4_Architecture_D_KD_Submission.zip
ZIP root contents:   prediction.json
